# Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install tqdm

from pathlib import Path
from PIL import Image
import numpy as np
import shutil
from tqdm import tqdm

IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".tif",".tiff",".webp"}

def _list_images(images_root: str):
    images_root = Path(images_root)
    return sorted([p for p in images_root.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS])

def _save_rgb_uint8(img_rgb_uint8: np.ndarray, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(img_rgb_uint8).save(str(out_path))

def _copy_label(dataset_root: str, out_root: str, rel_img_path: Path):
    # rel_img_path is relative to <dataset_root>/images
    src_lbl = Path(dataset_root) / "labels" / rel_img_path.with_suffix(".txt")
    dst_lbl = Path(out_root) / "labels" / rel_img_path.with_suffix(".txt")
    dst_lbl.parent.mkdir(parents=True, exist_ok=True)
    if src_lbl.exists():
        shutil.copy2(src_lbl, dst_lbl)

def build_defended_dataset(dataset_root: str, out_root: str, defend_fn, *, overwrite=False):
    """
    Applies `defend_fn(img_rgb_uint8)->img_rgb_uint8` to every image in <dataset_root>/images/**/*
    Saves into <out_root>/images/**/* and copies matching labels from <dataset_root>/labels/**/*
    """
    dataset_root = str(dataset_root)
    out_root = str(out_root)

    in_images = Path(dataset_root) / "images"
    if not in_images.exists():
        raise FileNotFoundError(f"Can't find images folder: {in_images}")

    if overwrite and Path(out_root).exists():
        shutil.rmtree(out_root)

    img_paths = _list_images(in_images)
    print(f"Found {len(img_paths)} images under: {in_images}")

    (Path(out_root) / "images").mkdir(parents=True, exist_ok=True)
    (Path(out_root) / "labels").mkdir(parents=True, exist_ok=True)

    ok, fail = 0, 0
    for p in tqdm(img_paths, desc=f"Defending -> {Path(out_root).name}"):
        rel = p.relative_to(in_images)               # e.g. test/rgb_0001.png
        out_img = Path(out_root) / "images" / rel
        try:
            img = np.array(Image.open(p).convert("RGB"), dtype=np.uint8)
            defended = defend_fn(img)
            defended = np.clip(defended, 0, 255).astype(np.uint8)

            _save_rgb_uint8(defended, out_img)
            _copy_label(dataset_root, out_root, rel)
            ok += 1
        except Exception as e:
            fail += 1

    print(f"Done. OK={ok}, FAIL={fail}")


In [ ]:
# Define path to your YOLOv8 model
YOLO_MODEL_PATH = "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt"

In [ ]:
# Install Ultralytics YOLOv8
!pip install ultralytics

!pip install kornia
!pip install torch torchvision matplotlib

# Load trained YOLOv8 model


In [ ]:
import torch

# Ultralytics likes device as int for GPU (0) or "cpu"
device = 0 if torch.cuda.is_available() else "cpu"
print("Device:", device)


In [ ]:
from ultralytics import YOLO

# Load trained YOLOv8 model
model = YOLO(YOLO_MODEL_PATH)
model.fuse()

model.model.to(device)
model.to(device)


In [ ]:
!pip -q install ultralytics scikit-learn

import os, glob, shutil, time, math
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

from ultralytics import YOLO


In [ ]:
import os, glob
from pathlib import Path

PATCHED_ROOT = Path("/content/drive/MyDrive/Patched Datasets/")
DEFENDED_ROOT = Path("/content/drive/MyDrive/Patched Datasets/Defended")

# auto-detect scenario folders (billboard01.. etc)
SCENARIOS = sorted([p.name for p in PATCHED_ROOT.iterdir() if p.is_dir()])
print("Found scenarios:", SCENARIOS[:10], " ... total:", len(SCENARIOS))

def scenario_dirs(root: Path, scenario: str):
    img_dir = root / scenario / "images" / "test"
    lab_dir = root / scenario / "labels" / "test"
    return img_dir, lab_dir

# quick sanity check for first scenario
s0 = SCENARIOS[0]
img_dir, lab_dir = scenario_dirs(PATCHED_ROOT, s0)
print("Example scenario:", s0)
print("Images dir exists:", img_dir.exists(), img_dir)
print("Labels dir exists:", lab_dir.exists(), lab_dir)
print("Num images:", len(list(img_dir.glob("*.png"))) + len(list(img_dir.glob("*.jpg"))) + len(list(img_dir.glob("*.jpeg"))))
print("Num labels:", len(list(lab_dir.glob("*.txt"))))


In [ ]:
# ===========================
# INPUT CELL: define `patched` (RGB uint8) from a file path
# ===========================
from PIL import Image
import numpy as np

img_path = "/content/drive/MyDrive/Patched Datasets/patched_dataset1/images/test/rgb_0001.png"
patched = np.array(Image.open(img_path).convert("RGB"), dtype=np.uint8)

print("patched:", patched.shape, patched.dtype)


## Defense 1: Simple LGS

In [ ]:
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt

def lgs_defense(
    image_tensor,              # (C,H,W) in [0,1], RGB
    sigma=100,                 # blur strength
    threshold=0.25,            # mask threshold (for normalized hf map)
    morph_kernel_size=3,       # clean mask
    soft_mask=True,            # blend instead of hard replace
    mask_mode="laplacian",     # "laplacian" (recommended) or "sobel"
    debug=False
):
    assert image_tensor.ndim == 3, "Expected (C,H,W)"
    C,H,W = image_tensor.shape
    assert C == 3, "Expected RGB"

    # tensor -> numpy RGB float [0,1]
    img = image_tensor.detach().float().clamp(0,1).cpu().numpy().transpose(1,2,0)

    # grayscale uint8 for gradients
    gray = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_RGB2GRAY)

    # --- build "high-frequency" map ---
    if mask_mode == "sobel":
        gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
        hf = cv2.magnitude(gx, gy)
    elif mask_mode == "laplacian":
        hf = cv2.Laplacian(gray, cv2.CV_32F, ksize=3)
        hf = np.abs(hf)
    else:
        raise ValueError("mask_mode must be 'laplacian' or 'sobel'")

    # normalize to [0,1]
    hf_norm = hf / (hf.max() + 1e-8)

    # --- binary mask from threshold ---
    mask = (hf_norm > threshold).astype(np.uint8)

    # clean mask (optional)
    if morph_kernel_size and morph_kernel_size > 0:
        k = np.ones((morph_kernel_size, morph_kernel_size), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)

    # convert to float mask
    mask_f = mask.astype(np.float32)

    # OPTIONAL: soften mask edges for nicer blending
    if soft_mask:
        mask_f = cv2.GaussianBlur(mask_f, (0,0), sigmaX=1.0, sigmaY=1.0)
        mask_f = np.clip(mask_f, 0, 1)

    # blur full RGB image (float)
    blurred = cv2.GaussianBlur(img, (0,0), sigmaX=sigma, sigmaY=sigma)

    # blend: blur where mask=1, keep original where mask=0
    out = img*(1-mask_f[...,None]) + blurred*mask_f[...,None]
    out = np.clip(out, 0, 1)

    if debug:
        plt.figure(figsize=(16,4))
        plt.subplot(1,4,1); plt.imshow(img); plt.title("Original"); plt.axis("off")
        plt.subplot(1,4,2); plt.imshow(hf_norm, cmap="hot"); plt.title(f"HF map ({mask_mode})"); plt.axis("off")
        plt.subplot(1,4,3); plt.imshow(mask_f, cmap="gray"); plt.title("Mask"); plt.axis("off")
        plt.subplot(1,4,4); plt.imshow(out); plt.title("LGS output"); plt.axis("off")
        plt.tight_layout(); plt.show()

    out_t = torch.from_numpy(out.transpose(2,0,1)).float().to(image_tensor.device)
    return out_t.clamp(0,1)


In [ ]:
import torch
import matplotlib.pyplot as plt

# numpy RGB uint8 -> torch (C,H,W) float in [0,1]
patched_t = torch.from_numpy(patched).permute(2,0,1).float() / 255.0

# LGS defend (start with these)
defended_lgs_t = lgs_defense(
    patched_t,
    sigma=100.0,
    threshold=0.08,
    morph_kernel_size=3,
    soft_mask=True,
    mask_mode="laplacian",
    debug=True
)

# back to numpy uint8 for YOLO
defended_lgs = (defended_lgs_t.permute(1,2,0).cpu().numpy() * 255).astype(np.uint8)

# Alias to the names your YOLO cell expects
defended_t = defended_lgs_t

plt.figure(figsize=(12,5))
plt.subplot(1,2,1); plt.imshow(patched); plt.title("PATCHED"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(defended_lgs); plt.title("DEFENDED (LGS)"); plt.axis("off")
plt.show()


In [ ]:
# ===========================
# YOLO eval (DEMO-STYLE) on PATCHED vs DEFENDED tensors
# Requires: model, patched_t (3,H,W) in [0,1], defended_t (3,H,W) in [0,1]
# ===========================

import cv2
import matplotlib.pyplot as plt

model.model.eval()

# demo-style defaults (uses model default conf/iou/max_det)
r_p = model(patched_t.unsqueeze(0), verbose=False)[0]
r_d = model(defended_t.unsqueeze(0), verbose=False)[0]

n_p = 0 if r_p.boxes is None else len(r_p.boxes)
n_d = 0 if r_d.boxes is None else len(r_d.boxes)
print(f"Detections (defaults ~conf=0.25): patched={n_p} | defended={n_d}")

vis_p = cv2.cvtColor(r_p.plot(), cv2.COLOR_BGR2RGB)
vis_d = cv2.cvtColor(r_d.plot(), cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14,6))
plt.subplot(1,2,1); plt.imshow(vis_p); plt.title(f"YOLO on PATCHED (n={n_p})"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(vis_d); plt.title(f"YOLO on DEFENDED (n={n_d})"); plt.axis("off")
plt.show()


In [ ]:
import numpy as np
import torch

DATASET_ROOT = "/content/drive/MyDrive/Patched Datasets/patched_dataset1"
OUT_ROOT     = "/content/drive/MyDrive/Patched Datasets/Defended/patched_dataset1_defended_LGS"

def defend_lgs_full_image(img_rgb_uint8: np.ndarray) -> np.ndarray:
    patched_t = torch.from_numpy(img_rgb_uint8).permute(2,0,1).float() / 255.0
    defended_t = lgs_defense(
        patched_t,
        sigma=50.0,
        threshold=0.08,
        morph_kernel_size=3,
        soft_mask=True,
        mask_mode="laplacian",
        debug=False
    )
    return (defended_t.permute(1,2,0).cpu().numpy() * 255.0).astype(np.uint8)

build_defended_dataset(DATASET_ROOT, OUT_ROOT, defend_lgs_full_image, overwrite=False)
print("✅ Saved:", OUT_ROOT)


## Defense 2: Patch Block defense follows a three-stage pipeline (Chunking → Separating → Mitigating)

## CHUNKING (extract chunks + features)

In [ ]:
!pip -q install scikit-learn opencv-python

import numpy as np
import cv2
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt

# ----- helpers -----
def mi_hist2d(a_uint8, b_uint8, bins=64, eps=1e-12):
    h2d, _, _ = np.histogram2d(
        a_uint8.ravel(), b_uint8.ravel(),
        bins=bins, range=[[0,256],[0,256]]
    )
    pxy = h2d / (h2d.sum() + eps)
    px = pxy.sum(axis=1, keepdims=True)
    py = pxy.sum(axis=0, keepdims=True)
    denom = px @ py
    nz = pxy > 0
    return float((pxy[nz] * np.log((pxy[nz] + eps) / (denom[nz] + eps))).sum())

def entropy_uint8(a_uint8, bins=64, eps=1e-12):
    h, _ = np.histogram(a_uint8.ravel(), bins=bins, range=(0,256))
    p = h.astype(np.float32)
    p = p / (p.sum() + eps)
    p = p[p > 0]
    return float(-(p * np.log(p + eps)).sum())

def hf_energy(a_uint8):
    lap = cv2.Laplacian(a_uint8, cv2.CV_32F, ksize=3)
    return float(lap.var())

# ----- Stage 1: CHUNKING -----
def chunk_and_extract_features(img_rgb_uint8, k=50, stride=10, bins=64):
    """
    Returns:
      coords: list of (y,x) chunk top-left corners
      feats:  NxD features per chunk
    """
    H, W = img_rgb_uint8.shape[:2]

    hsv = cv2.cvtColor(img_rgb_uint8, cv2.COLOR_RGB2HSV)
    S = hsv[:, :, 1]  # saturation
    gray = cv2.cvtColor(img_rgb_uint8, cv2.COLOR_RGB2GRAY)

    coords = []
    feats = []

    for y in range(0, H - k + 1, stride):
        for x in range(0, W - k + 1, stride):
            patch = img_rgb_uint8[y:y+k, x:x+k, :]
            R, G, B = patch[:,:,0], patch[:,:,1], patch[:,:,2]

            # PatchBlock-motivated channel-dependency signal
            mi_rgb = mi_hist2d(R, G, bins=bins) + mi_hist2d(G, B, bins=bins) + mi_hist2d(R, B, bins=bins)

            # Localized MI chunk <-> neighbors (Eq.5 idea), computed on gray
            P = gray[y:y+k, x:x+k]
            nbr = []
            for dy, dx in [(-stride,0),(stride,0),(0,-stride),(0,stride)]:
                y2, x2 = y+dy, x+dx
                if 0 <= y2 <= H-k and 0 <= x2 <= W-k:
                    N = gray[y2:y2+k, x2:x2+k]
                    nbr.append(mi_hist2d(P, N, bins=bins))
            mi_local = float(np.mean(nbr)) if nbr else 0.0

            # Extra “your-patch” signature features
            S_patch = S[y:y+k, x:x+k]
            sat_mean = float(S_patch.mean()) / 255.0
            ent_s = entropy_uint8(S_patch, bins=bins)
            hf_s = hf_energy(S_patch)

            coords.append((y, x))
            feats.append([mi_rgb, mi_local, sat_mean, ent_s, hf_s])

    return coords, np.array(feats, dtype=np.float32)


## SEPARATING (outlier detection + keep patch-like connected component)

In [ ]:
# ----- Stage 2: SEPARATING -----
def separate_anomalous_chunks(coords, feats, img_shape_hw, k=50, stride=10, outlier_frac=0.02):
    """
    Returns:
      keep_idx: indices of chunks to mitigate (selected patch-like component)
      scores: anomaly scores (higher => more anomalous)
    """
    H, W = img_shape_hw
    n = len(coords)
    if n == 0:
        return np.array([], dtype=int), None

    # PatchBlock Algorithm 1 uses sample size s = 0.3 * |X|
    max_samples = max(2, int(0.3 * n))

    iso = IsolationForest(
        n_estimators=200,
        max_samples=max_samples,
        contamination=outlier_frac,
        random_state=0,
        n_jobs=-1
    )
    iso.fit(feats)
    scores = -iso.decision_function(feats)

    # pick top fraction
    r = max(1, int(outlier_frac * n))
    top = np.argsort(scores)[-r:]

    # grid mask for connected components
    gh = (H - k) // stride + 1
    gw = (W - k) // stride + 1
    mask = np.zeros((gh, gw), dtype=np.uint8)
    for i in top:
        y, x = coords[i]
        mask[y // stride, x // stride] = 1

    num, labels = cv2.connectedComponents(mask)
    if num <= 1:
        return top.astype(int), scores

    # choose component most “patch-like”: high sat_mean + high hf_s + high entropy
    # feats columns: [mi_rgb, mi_local, sat_mean, ent_s, hf_s]
    best_lab, best_score = 1, -1e18
    for lab in range(1, num):
        idxs = []
        for i in top:
            y, x = coords[i]
            if labels[y // stride, x // stride] == lab:
                idxs.append(i)
        if not idxs:
            continue
        f = feats[idxs]
        comp_score = (f[:,2].mean()*3.0) + (f[:,4].mean()*0.002) + (f[:,3].mean()*1.0)
        if comp_score > best_score:
            best_score = comp_score
            best_lab = lab

    keep = []
    for i in top:
        y, x = coords[i]
        if labels[y // stride, x // stride] == best_lab:
            keep.append(i)

    return np.array(keep, dtype=int), scores

def draw_boxes(img_rgb, coords, idxs, k):
    vis = img_rgb.copy()
    for i in idxs:
        y, x = coords[i]
        cv2.rectangle(vis, (x, y), (x+k, y+k), (255, 0, 0), 2)
    return vis


In [ ]:
# ===========================
# RUN (Stage 1 + Stage 2)
# This cell CREATES: patched, coords, feats, keep_idx
# ===========================

from PIL import Image
import numpy as np

# 1) Load ONE patched image (edit img_path)
img_path = "/content/drive/MyDrive/Patched Datasets/patched_dataset1/images/test/rgb_0002.png"

# define `patched` exactly (RGB uint8)
patched = np.array(Image.open(img_path).convert("RGB"), dtype=np.uint8)

# parameters
K = 50
STRIDE = 10
OUTLIER_FRAC = 0.02
BINS = 64

# 2) Stage 1: Chunking (must exist from your Stage 1 cell)
coords, feats = chunk_and_extract_features(patched, k=K, stride=STRIDE, bins=BINS)

# 3) Stage 2: Separating
H, W = patched.shape[:2]
keep_idx, scores = separate_anomalous_chunks(coords, feats, (H, W), k=K, stride=STRIDE, outlier_frac=OUTLIER_FRAC)

print("Image loaded:", img_path)
print("Total chunks:", len(coords))
print("Selected chunks:", len(keep_idx))


In [ ]:
boxed = draw_boxes(patched, coords, keep_idx, k=K)

plt.figure(figsize=(8,6))
plt.imshow(boxed)
plt.title(f"Step 2 output: Selected anomalous chunks (n={len(keep_idx)})")
plt.axis("off")
plt.show()


## MITIGATING (mitigate selected chunks + superimpose in place)

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch

# ----- Stage 3: MITIGATING -----
def svd_lowrank_rgb_matrix(patch_rgb_float, info=0.85):
    k = patch_rgb_float.shape[0]
    M = patch_rgb_float.reshape(k, -1)  # (k, k*3)
    U, S, Vt = np.linalg.svd(M, full_matrices=False)
    cum = np.cumsum(S) / (np.sum(S) + 1e-12)
    r = int(np.searchsorted(cum, info) + 1)
    Mr = (U[:, :r] * S[:r]) @ Vt[:r, :]
    return Mr.reshape(k, k, 3).astype(np.float32)

def mitigate_and_superimpose(img_rgb_uint8, coords, keep_idx, k=50, mitigation="svd", info=0.85):
    """
    Returns defended image (uint8).
    Superimpose mitigated chunks in place (paper-style).
    """
    out = img_rgb_uint8.astype(np.float32).copy()

    for i in keep_idx:
        y, x = coords[i]
        patch = out[y:y+k, x:x+k, :]

        if mitigation == "svd":
            den = svd_lowrank_rgb_matrix(patch, info=info)
        elif mitigation == "blur":
            den = cv2.GaussianBlur(patch.astype(np.uint8), (11,11), 0).astype(np.float32)
        else:
            raise ValueError("mitigation must be 'svd' or 'blur'")

        out[y:y+k, x:x+k, :] = den  # superimpose in place

    return np.clip(out, 0, 255).astype(np.uint8)


In [ ]:
# ===========================
# RUN PatchBlock (Stages 1-3) on ONE image
# Requires: patched (RGB uint8), chunk_and_extract_features, separate_anomalous_chunks, draw_boxes
# ===========================
K = 50
STRIDE = 10
OUTLIER_FRAC = 0.02
BINS = 64

# Stage 1: Chunking
coords, feats = chunk_and_extract_features(patched, k=K, stride=STRIDE, bins=BINS)

# Stage 2: Separating
H, W = patched.shape[:2]
keep_idx, scores = separate_anomalous_chunks(coords, feats, (H, W), k=K, stride=STRIDE, outlier_frac=OUTLIER_FRAC)

# Stage 3: Mitigating (set mitigation="svd" for paper-style; "blur" for stronger ablation)
MITIGATION = "svd"   # change to "blur" if you want the stronger visual effect
INFO = 0.65
defended = mitigate_and_superimpose(patched, coords, keep_idx, k=K, mitigation=MITIGATION, info=INFO)

# Visualize patch localization + defended image
boxed = draw_boxes(patched, coords, keep_idx, k=K)
plt.figure(figsize=(14,6))
plt.subplot(1,2,1); plt.imshow(boxed);   plt.title(f"Selected component (n={len(keep_idx)})"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(defended);plt.title(f"DEFENDED ({MITIGATION}, info={INFO})"); plt.axis("off")
plt.show()

In [ ]:
# ===========================
# YOLO EVAL (DEMO-STYLE) on PATCHED vs DEFENDED tensors
# ===========================
device = next(model.model.parameters()).device
model.model.eval()

patched_t  = torch.from_numpy(patched).permute(2,0,1).float().to(device) / 255.0
defended_t = torch.from_numpy(defended).permute(2,0,1).float().to(device) / 255.0

r_p = model(patched_t.unsqueeze(0),  verbose=False)[0]
r_d = model(defended_t.unsqueeze(0), verbose=False)[0]

n_p = 0 if r_p.boxes is None else len(r_p.boxes)
n_d = 0 if r_d.boxes is None else len(r_d.boxes)
print(f"Detections (defaults ~conf=0.25): patched={n_p} | defended={n_d}")

vis_p = cv2.cvtColor(r_p.plot(), cv2.COLOR_BGR2RGB)
vis_d = cv2.cvtColor(r_d.plot(), cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14,6))
plt.subplot(1,2,1); plt.imshow(vis_p); plt.title(f"YOLO on PATCHED (n={n_p})"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(vis_d); plt.title(f"YOLO on DEFENDED (n={n_d})"); plt.axis("off")
plt.show()

In [ ]:
import numpy as np

DATASET_ROOT = "/content/drive/MyDrive/Patched Datasets/patched_dataset1"
OUT_ROOT     = "/content/drive/MyDrive/Patched Datasets/Defended/patched_dataset1_defended_PatchBlock_SVD"

K = 50
STRIDE = 10
OUTLIER_FRAC = 0.02
BINS = 64
INFO = 0.65

def defend_patchblock_svd(img_rgb_uint8: np.ndarray) -> np.ndarray:
    coords, feats = chunk_and_extract_features(img_rgb_uint8, k=K, stride=STRIDE, bins=BINS)
    H, W = img_rgb_uint8.shape[:2]
    keep_idx, _ = separate_anomalous_chunks(
        coords, feats, (H, W),
        k=K, stride=STRIDE, outlier_frac=OUTLIER_FRAC
    )
    return mitigate_and_superimpose(
        img_rgb_uint8, coords, keep_idx,
        k=K, mitigation="svd", info=INFO
    )

build_defended_dataset(DATASET_ROOT, OUT_ROOT, defend_patchblock_svd, overwrite=False)
print("✅ Saved:", OUT_ROOT)


In [ ]:
# ===========================
# PAPER-IDENTICAL EVAL (Cell 2 clone)
# - Per-scenario eval (mAP + FP/img mean + Det/img mean)
# - Mean ± std computed across scenario-level values
# - Greedy class-aware matching for FP counting
# ===========================

!pip -q install ultralytics pyyaml tqdm

import os, glob, re, yaml
import numpy as np
import pandas as pd
from PIL import Image
import torch
from ultralytics import YOLO

# ---------------------------
# EDIT THESE 2 THINGS ONLY
# ---------------------------
MODEL_PATH = "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt"

# Each method root MUST have:
#   <ROOT>/<scenario>/images/test/*.png|jpg|jpeg
#   <ROOT>/<scenario>/labels/test/*.txt
METHODS = [
    ("PATCHED",   "/content/drive/MyDrive/RANDOM/billr_patched_scenarios"),
    ("DEF_LGS",   "/content/drive/MyDrive/RANDOM/billr_defended_lgs_scenarios"),
    ("DEF_PB_SVD","/content/drive/MyDrive/RANDOM/billr_defended_pb_svd_scenarios"),
    ("DEF_PB_LGS","/content/drive/MyDrive/RANDOM/billr_defended_pb_lgs_scenarios"),
]

# ---------------------------
# SAME SETTINGS AS YOUR CELL 2
# ---------------------------
SCENARIOS  = [f"billboard{i:02d}" for i in range(1, 10)]
CONFS      = [0.001, 0.3, 0.5]      # use whatever you want, but keep consistent across methods
IMGSZ      = 640
NMS_IOU    = 0.7
IOU_MATCH  = 0.50
EXTS       = ("*.png", "*.jpg", "*.jpeg")  # EXACTLY like your Cell 2

# ---------------------------
# Model
# ---------------------------
model = YOLO(MODEL_PATH)

# names for yaml (optional)
names = getattr(model.model, "names", None) or getattr(model, "names", None)
if isinstance(names, dict):
    names_list = [names[i] for i in sorted(names.keys())]
elif isinstance(names, list):
    names_list = names
else:
    names_list = None

def write_yaml(root_dir, out_path):
    # EXACT Cell 2 pattern: train/val/test all point to images/test
    d = {"path": root_dir, "train": "images/test", "val": "images/test", "test": "images/test"}
    if names_list is not None:
        d["names"] = names_list
    with open(out_path, "w") as f:
        yaml.safe_dump(d, f, sort_keys=False)

# ---------------------------
# Robust label parsing (same as your Cell 2)
# ---------------------------
_num_re = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

def yolo_label_to_xyxy(line, w, h):
    nums = _num_re.findall(line)
    if len(nums) < 5:
        return None
    cls = int(float(nums[0]))
    xc, yc, bw, bh = map(float, nums[1:5])
    x1 = (xc - bw/2) * w
    y1 = (yc - bh/2) * h
    x2 = (xc + bw/2) * w
    y2 = (yc + bh/2) * h
    return cls, np.array([x1, y1, x2, y2], dtype=np.float32)

def iou_xyxy(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter = inter_w * inter_h
    area_a = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    area_b = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])
    union = area_a + area_b - inter + 1e-9
    return inter / union

@torch.no_grad()
def scenario_fp_det_means(images_dir, labels_dir, conf):
    img_paths = []
    for e in EXTS:
        img_paths.extend(glob.glob(os.path.join(images_dir, e)))
    img_paths = sorted(img_paths)
    if not img_paths:
        return None

    fp_list = []
    det_list = []

    for p in img_paths:
        img = Image.open(p).convert("RGB")
        w, h = img.size
        stem = os.path.splitext(os.path.basename(p))[0]
        lab_path = os.path.join(labels_dir, stem + ".txt")

        gts = []
        if os.path.exists(lab_path):
            with open(lab_path, "r") as f:
                for line in f:
                    r = yolo_label_to_xyxy(line, w, h)
                    if r is not None:
                        gts.append(r)

        pred = model.predict(
            source=np.array(img),
            conf=conf,
            iou=NMS_IOU,
            imgsz=IMGSZ,
            verbose=False
        )
        boxes = pred[0].boxes
        if boxes is None or len(boxes) == 0:
            fp_list.append(0)
            det_list.append(0)
            continue

        det_xyxy = boxes.xyxy.cpu().numpy()
        det_cls  = boxes.cls.cpu().numpy().astype(int)

        det_count = len(det_xyxy)
        fp_count = 0

        # Greedy, class-aware matching (EXACT)
        used_gt = set()
        for j in range(det_count):
            pj = det_xyxy[j]
            cj = det_cls[j]

            best_iou, best_k = 0.0, None
            for k, (cg, gg) in enumerate(gts):
                if k in used_gt:
                    continue
                if cg != cj:
                    continue
                i = iou_xyxy(pj, gg)
                if i > best_iou:
                    best_iou, best_k = i, k

            if best_k is not None and best_iou >= IOU_MATCH:
                used_gt.add(best_k)  # TP
            else:
                fp_count += 1        # FP

        fp_list.append(fp_count)
        det_list.append(det_count)

    fp_arr = np.array(fp_list, dtype=float)
    det_arr = np.array(det_list, dtype=float)
    return float(fp_arr.mean()), float(det_arr.mean()), len(img_paths)

# ---------------------------
# RUN: per-scenario + mean±std across scenarios (EXACT)
# ---------------------------
all_rows = []
all_agg  = []

for method_name, method_root in METHODS:
    print("\n==============================")
    print("METHOD:", method_name)
    print("ROOT:", method_root)

    rows = []

    for scenario in SCENARIOS:
        root = os.path.join(method_root, scenario)
        images_dir = os.path.join(root, "images", "test")
        labels_dir = os.path.join(root, "labels", "test")

        if not os.path.exists(images_dir):
            print("Skipping missing:", method_name, scenario)
            continue

        yaml_path = f"/content/tmp_{method_name}_{scenario}.yaml"
        write_yaml(root, yaml_path)

        for conf in CONFS:
            # Ultralytics mAP
            val_res = model.val(data=yaml_path, conf=conf, iou=NMS_IOU, imgsz=IMGSZ, verbose=False)
            map5095 = float(val_res.box.map) * 100.0
            map50   = float(val_res.box.map50) * 100.0

            fp_det = scenario_fp_det_means(images_dir, labels_dir, conf)
            if fp_det is None:
                continue
            fp_mean, det_mean, n_imgs = fp_det

            rows.append({
                "method": method_name,
                "scenario": scenario,
                "conf": conf,
                "mAP50-95 (%)": map5095,
                "mAP50 (%)": map50,
                "FP/img mean": fp_mean,
                "Det/img mean": det_mean,
                "images": n_imgs
            })

    df = pd.DataFrame(rows).sort_values(["conf", "scenario"]).reset_index(drop=True)
    print("\n=== PER-SCENARIO RESULTS:", method_name, "===")
    display(df)

    # Mean ± std across scenarios (std over scenario-level values) — EXACT
    agg_rows = []
    for conf in CONFS:
        dconf = df[df["conf"] == conf].copy()
        if len(dconf) == 0:
            continue
        agg_rows.append({
            "method": method_name,
            "conf": conf,
            "mAP50-95 mean": dconf["mAP50-95 (%)"].mean(),
            "mAP50-95 std":  dconf["mAP50-95 (%)"].std(ddof=1),
            "mAP50 mean":    dconf["mAP50 (%)"].mean(),
            "mAP50 std":     dconf["mAP50 (%)"].std(ddof=1),
            "FP/img mean":   dconf["FP/img mean"].mean(),
            "FP/img std":    dconf["FP/img mean"].std(ddof=1),
            "Det/img mean":  dconf["Det/img mean"].mean(),
            "Det/img std":   dconf["Det/img mean"].std(ddof=1),
            "scenarios": int(dconf["scenario"].nunique())
        })

    df_agg = pd.DataFrame(agg_rows).sort_values("conf").reset_index(drop=True)
    print("\n=== MEAN ± STD ACROSS SCENARIOS:", method_name, "===")
    display(df_agg)

    # save per-method CSVs
    out1 = os.path.join(method_root, f"{method_name}_per_scenario_results.csv")
    out2 = os.path.join(method_root, f"{method_name}_mean_std_across_scenarios.csv")
    try:
        df.to_csv(out1, index=False)
        df_agg.to_csv(out2, index=False)
        print("✅ Saved:", out1)
        print("✅ Saved:", out2)
    except Exception as e:
        print("⚠️ Could not save CSVs to method_root (permissions/path):", e)

    all_rows.append(df)
    all_agg.append(df_agg)

# Combined outputs (all methods together)
df_all = pd.concat(all_rows, ignore_index=True) if len(all_rows) else pd.DataFrame()
df_all_agg = pd.concat(all_agg, ignore_index=True) if len(all_agg) else pd.DataFrame()

print("\n==============================")
print("=== ALL METHODS: PER-SCENARIO (combined) ===")
display(df_all)

print("\n=== ALL METHODS: MEAN ± STD (combined) ===")
display(df_all_agg)

In [ ]:
# ===========================
# FINAL: SVD SWEEP EVALUATION (5 info values): PATCHED vs (PatchBlock + SVD)
# - Caches PatchBlock localization once
# - Builds defended datasets per info (overwrite=True)
# - Evaluates same metrics as before
# ===========================

!pip -q install ultralytics pyyaml tqdm

import os, glob, shutil, re, yaml
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
from ultralytics import YOLO

# ---------------------------
# MUST-SET PATHS
# ---------------------------
MODEL_PATH   = "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt"
PATCHED_ROOT = "/content/drive/MyDrive/Patched Datasets/patched_dataset1"  # images/<split>, labels/<split>

# ---------------------------
# PatchBlock params (same as your pipeline)
# ---------------------------
K = 50
STRIDE = 10
OUTLIER_FRAC = 0.02
BINS = 64

# ---------------------------
# Evaluation params
# ---------------------------
CONFS = [0.001, 0.3, 0.5]
IMGSZ = 640
NMS_IOU = 0.7
IOU_MATCH = 0.50
OVERLOAD_T = 50

# ---------------------------
# SVD sweep: 5 info values
# NOTE: Lower info => stronger SVD (more aggressive)
# Recommended "paper curve" set (weak -> knee -> safe -> strong)
# ---------------------------
SVD_INFOS = [0.85, 0.75, 0.65, 0.55, 0.45]  # <-- 5 values

# Optional: force max rank for all configs (None keeps automatic rank)
R_MAX = None   # e.g. 3 or 2 if you want a global cap; otherwise keep None

# Where to build temporary defended datasets
TMP_BASE = "/content/tmp_svd_sweep"

# (Recommended) clean old temp results so nothing stale remains
if os.path.exists(TMP_BASE):
    shutil.rmtree(TMP_BASE)
os.makedirs(TMP_BASE, exist_ok=True)

# ---------------------------
# Sanity: must have Stage1/2 funcs already
# ---------------------------
for fn in ["chunk_and_extract_features", "separate_anomalous_chunks"]:
    if fn not in globals():
        raise NameError(f"Missing `{fn}`. Run your PatchBlock Stage 1/2 definition cells first.")

def detect_split(root_dir):
    for s in ["test", "val", "valid", "train"]:
        if os.path.exists(os.path.join(root_dir, "images", s)) and os.path.exists(os.path.join(root_dir, "labels", s)):
            return s
    return None

split = detect_split(PATCHED_ROOT)
if split is None:
    raise FileNotFoundError(
        f"PATCHED_ROOT has no images/{{test|val|valid|train}} and labels/{{...}}:\n{PATCHED_ROOT}"
    )

images_dir = os.path.join(PATCHED_ROOT, "images", split)
labels_dir = os.path.join(PATCHED_ROOT, "labels", split)

img_paths = []
for ext in ("*.png","*.jpg","*.jpeg","*.webp"):
    img_paths += glob.glob(os.path.join(images_dir, ext))
img_paths = sorted(img_paths)
if not img_paths:
    raise RuntimeError(f"No images found in {images_dir}")

print("MODEL:", MODEL_PATH)
print("PATCHED_ROOT:", PATCHED_ROOT)
print("split:", split, "| images:", len(img_paths))
print("SVD_INFOS (5):", SVD_INFOS, "| R_MAX:", R_MAX)
print("TMP_BASE:", TMP_BASE)

# ---------------------------
# Model
# ---------------------------
model = YOLO(MODEL_PATH)

names = getattr(model.model, "names", None) or getattr(model, "names", None)
if isinstance(names, dict):
    names_list = [names[i] for i in sorted(names.keys())]
elif isinstance(names, list):
    names_list = names
else:
    names_list = None

# ---------------------------
# Helpers: labels + IoU (robust numeric parsing)
# ---------------------------
_num_re = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

def yolo_label_to_xyxy(line, w, h):
    nums = _num_re.findall(line)
    if len(nums) < 5:
        return None
    cls = int(float(nums[0]))
    xc, yc, bw, bh = map(float, nums[1:5])
    x1 = (xc - bw/2) * w
    y1 = (yc - bh/2) * h
    x2 = (xc + bw/2) * w
    y2 = (yc + bh/2) * h
    return cls, np.array([x1, y1, x2, y2], dtype=np.float32)

def iou_xyxy(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    iw = max(0.0, x2 - x1)
    ih = max(0.0, y2 - y1)
    inter = iw * ih
    area_a = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    area_b = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])
    return inter / (area_a + area_b - inter + 1e-9)

def write_yaml(root_dir, out_path):
    d = {"path": root_dir, "train": f"images/{split}", "val": f"images/{split}", "test": f"images/{split}"}
    if names_list is not None:
        d["names"] = names_list
    with open(out_path, "w") as f:
        yaml.safe_dump(d, f, sort_keys=False)

# ---------------------------
# PatchBlock localization cache (compute ONCE)
# ---------------------------
print("\nPrecomputing PatchBlock localization once...")
cache = {}  # stem -> coords, keep_idx

for p in tqdm(img_paths, desc="PatchBlock locate", leave=False):
    stem = os.path.splitext(os.path.basename(p))[0]
    img = np.array(Image.open(p).convert("RGB"), dtype=np.uint8)
    H, W = img.shape[:2]
    coords, feats = chunk_and_extract_features(img, k=K, stride=STRIDE, bins=BINS)
    keep_idx, _ = separate_anomalous_chunks(coords, feats, (H, W), k=K, stride=STRIDE, outlier_frac=OUTLIER_FRAC)
    cache[stem] = {"coords": coords, "keep_idx": keep_idx}

print("✅ Cached:", len(cache), "images")

# ---------------------------
# Stage 3: SVD mitigation (with optional rank cap)
# ---------------------------
def svd_lowrank_rgb_matrix(patch_rgb_float, info=0.85, r_max=None):
    k = patch_rgb_float.shape[0]
    M = patch_rgb_float.reshape(k, -1)  # (k, k*3)
    U, S, Vt = np.linalg.svd(M, full_matrices=False)
    cum = np.cumsum(S) / (np.sum(S) + 1e-12)
    r = int(np.searchsorted(cum, info) + 1)
    r = max(1, r)
    if r_max is not None:
        r = min(r, int(r_max))
    Mr = (U[:, :r] * S[:r]) @ Vt[:r, :]
    return Mr.reshape(k, k, 3).astype(np.float32), r

def mitigate_svd_in_windows(img_rgb_uint8, coords, keep_idx, k=50, info=0.85, r_max=None):
    out = img_rgb_uint8.astype(np.float32).copy()
    ranks = []
    for i in keep_idx:
        y, x = coords[i]
        patch = out[y:y+k, x:x+k, :]
        if patch.shape[0] != k or patch.shape[1] != k:
            continue
        den, r = svd_lowrank_rgb_matrix(patch, info=info, r_max=r_max)
        out[y:y+k, x:x+k, :] = den
        ranks.append(r)
    return np.clip(out, 0, 255).astype(np.uint8), ranks

def build_defended_dataset_svd(tag, info, r_max, overwrite=True):
    out_root  = os.path.join(TMP_BASE, tag)
    out_images = os.path.join(out_root, "images", split)
    out_labels = os.path.join(out_root, "labels", split)

    if overwrite and os.path.exists(out_root):
        shutil.rmtree(out_root)

    os.makedirs(out_images, exist_ok=True)
    os.makedirs(out_labels, exist_ok=True)

    # copy labels
    for lab in glob.glob(os.path.join(labels_dir, "*.txt")):
        shutil.copy2(lab, os.path.join(out_labels, os.path.basename(lab)))

    all_ranks = []
    for p in tqdm(img_paths, desc=f"Defend {tag}", leave=False):
        stem = os.path.splitext(os.path.basename(p))[0]
        img = np.array(Image.open(p).convert("RGB"), dtype=np.uint8)

        coords   = cache[stem]["coords"]
        keep_idx = cache[stem]["keep_idx"]

        if len(keep_idx) == 0:
            defended = img
            ranks = []
        else:
            defended, ranks = mitigate_svd_in_windows(img, coords, keep_idx, k=K, info=info, r_max=r_max)

        all_ranks.extend(ranks)
        Image.fromarray(defended).save(os.path.join(out_images, os.path.basename(p)))

    if len(all_ranks) > 0:
        rank_stats = (float(np.mean(all_ranks)), int(np.min(all_ranks)), int(np.max(all_ranks)))
    else:
        rank_stats = (0.0, 0, 0)

    return out_root, rank_stats

# ---------------------------
# Counts metrics (FP/Det/TP/FN + overload etc.)
# ---------------------------
def compute_counts_metrics(root_dir, conf):
    img_dir = os.path.join(root_dir, "images", split)
    lab_dir = os.path.join(root_dir, "labels", split)

    paths = []
    for ext in ("*.png","*.jpg","*.jpeg","*.webp"):
        paths += glob.glob(os.path.join(img_dir, ext))
    paths = sorted(paths)
    n_imgs = len(paths)

    det_list, fp_list, tp_list, fn_list = [], [], [], []
    overload_count = 0
    total_tp = total_fp = total_fn = 0

    with torch.no_grad():
        for p in paths:
            img = Image.open(p).convert("RGB")
            w, h = img.size
            stem = os.path.splitext(os.path.basename(p))[0]
            lab_path = os.path.join(lab_dir, stem + ".txt")

            gts = []
            if os.path.exists(lab_path):
                with open(lab_path, "r") as f:
                    for line in f:
                        r = yolo_label_to_xyxy(line, w, h)
                        if r is not None:
                            gts.append(r)

            pred = model.predict(source=np.array(img), conf=conf, iou=NMS_IOU, imgsz=IMGSZ, verbose=False)[0]
            boxes = pred.boxes

            if boxes is None or len(boxes) == 0:
                det_count = 0
                tp = 0
                fp = 0
                fn = len(gts)
            else:
                det_xyxy = boxes.xyxy.cpu().numpy()
                det_cls  = boxes.cls.cpu().numpy().astype(int)
                det_count = len(det_xyxy)

                if det_count >= OVERLOAD_T:
                    overload_count += 1

                used_gt = set()
                tp = 0
                fp = 0
                for j in range(det_count):
                    pj = det_xyxy[j]
                    cj = det_cls[j]

                    best_iou, best_k = 0.0, None
                    for k, (cg, gg) in enumerate(gts):
                        if k in used_gt:
                            continue
                        if cg != cj:
                            continue
                        i = iou_xyxy(pj, gg)
                        if i > best_iou:
                            best_iou, best_k = i, k

                    if best_k is not None and best_iou >= IOU_MATCH:
                        used_gt.add(best_k)
                        tp += 1
                    else:
                        fp += 1

                fn = max(0, len(gts) - tp)

            det_list.append(det_count)
            fp_list.append(fp)
            tp_list.append(tp)
            fn_list.append(fn)

            total_tp += tp
            total_fp += fp
            total_fn += fn

    det_arr = np.array(det_list, dtype=float)
    fp_arr  = np.array(fp_list, dtype=float)
    tp_arr  = np.array(tp_list, dtype=float)
    fn_arr  = np.array(fn_list, dtype=float)

    precision = total_tp / (total_tp + total_fp + 1e-9)
    recall    = total_tp / (total_tp + total_fn + 1e-9)

    return {
        "FP/img mean": float(fp_arr.mean()),
        "FP/img std":  float(fp_arr.std(ddof=1)) if n_imgs > 1 else 0.0,
        "Det/img mean": float(det_arr.mean()),
        "Det/img std":  float(det_arr.std(ddof=1)) if n_imgs > 1 else 0.0,
        "TP/img mean": float(tp_arr.mean()),
        "FN/img mean": float(fn_arr.mean()),
        "Precision": float(precision),
        "Recall": float(recall),
        "Overload@50 (%)": float(100.0 * overload_count / max(1, n_imgs)),
        "Det/img median": float(np.median(det_arr)),
        "Det/img p95": float(np.percentile(det_arr, 95)),
        "images": int(n_imgs),
    }

def eval_one_dataset(name, root_dir, conf):
    yaml_path = f"/content/tmp_eval_{re.sub('[^a-zA-Z0-9_]+','_',name)}.yaml"
    write_yaml(root_dir, yaml_path)
    val_res = model.val(data=yaml_path, conf=conf, iou=NMS_IOU, imgsz=IMGSZ, verbose=False)
    map5095 = float(val_res.box.map) * 100.0
    map50   = float(val_res.box.map50) * 100.0
    counts  = compute_counts_metrics(root_dir, conf)
    return {"dataset": name, "conf": conf, "mAP50-95 (%)": map5095, "mAP50 (%)": map50, **counts}

# ---------------------------
# Evaluate baseline PATCHED
# ---------------------------
rows = []
print("\n==============================")
print("EVALUATING baseline: PATCHED")
for conf in CONFS:
    r = eval_one_dataset("PATCHED", PATCHED_ROOT, conf)
    rows.append(r)
    print(f"[PATCHED] conf={conf:.3f} | mAP50-95={r['mAP50-95 (%)']:.2f}% mAP50={r['mAP50 (%)']:.2f}% | "
          f"FP/img={r['FP/img mean']:.2f}±{r['FP/img std']:.2f} Det/img={r['Det/img mean']:.2f}±{r['Det/img std']:.2f} | "
          f"P={r['Precision']:.3f} R={r['Recall']:.3f} | Overload@50={r['Overload@50 (%)']:.1f}%")

# ---------------------------
# Sweep 5 SVD infos 1-by-1
# ---------------------------
for info in SVD_INFOS:
    tag = f"DEF_PB_SVD_info{info:.2f}_r{('None' if R_MAX is None else int(R_MAX))}"
    print("\n==============================")
    print("Building defended dataset:", tag)

    out_root, rank_stats = build_defended_dataset_svd(tag, info=info, r_max=R_MAX, overwrite=True)
    avg_r, min_r, max_r = rank_stats
    print(f"SVD ranks: avg={avg_r:.2f}, min={min_r}, max={max_r}")

    print("EVALUATING:", tag)
    for conf in CONFS:
        r = eval_one_dataset(tag, out_root, conf)
        r["svd_info"] = info
        r["svd_r_max"] = R_MAX
        r["svd_rank_avg"] = avg_r
        r["svd_rank_min"] = min_r
        r["svd_rank_max"] = max_r
        rows.append(r)

        print(f"[{tag}] conf={conf:.3f} | mAP50-95={r['mAP50-95 (%)']:.2f}% mAP50={r['mAP50 (%)']:.2f}% | "
              f"FP/img={r['FP/img mean']:.2f}±{r['FP/img std']:.2f} Det/img={r['Det/img mean']:.2f}±{r['Det/img std']:.2f} | "
              f"P={r['Precision']:.3f} R={r['Recall']:.3f} | Overload@50={r['Overload@50 (%)']:.1f}%")

# ---------------------------
# Summary + "best" selection @ conf=0.001
# ---------------------------
df = pd.DataFrame(rows).sort_values(["conf", "dataset"]).reset_index(drop=True)

print("\n=== FINAL SUMMARY TABLE ===")
display(df)

target_conf = 0.001
dsub = df[(df["conf"] == target_conf) & (df["dataset"].str.startswith("DEF_PB_SVD"))].copy()
if len(dsub):
    dsub = dsub.sort_values(["Overload@50 (%)", "FP/img mean", "mAP50-95 (%)"], ascending=[True, True, False])
    print("\n=== BEST 5 SVD CONFIGS @ conf=0.001 (by overload -> FP -> mAP) ===")
    display(dsub.head(5))
else:
    print("\n(No defended rows found)")

out_csv = os.path.join(TMP_BASE, "svd_sweep_5info_results.csv")
df.to_csv(out_csv, index=False)
print("\n✅ Saved CSV:", out_csv)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------ Style ------------------
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def is_patched_series(df):
    return df["dataset"].astype(str).str.contains("PATCHED", regex=False)

def is_svd_series(df):
    return df["dataset"].astype(str).str.contains("DEF_PB_SVD", regex=False)

def label_info(row):
    return f"info={float(row['svd_info']):.2f}"

# helper: keep annotation text inside axes a bit better
def annotate_clamped(ax, x, y, text, dx=10, dy=10, fontsize=10):
    ax.annotate(
        text, (x, y),
        textcoords="offset points", xytext=(dx, dy),
        ha="left", va="bottom",
        fontsize=fontsize,
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.85),
        arrowprops=dict(arrowstyle="-", linewidth=0.8, alpha=0.6),
        zorder=10
    )

# ------------------ Common masks ------------------
pts_is_patched = is_patched_series(pts)
pts_is_def = ~pts_is_patched

# Threshold for "close ones" (only used to decide what to label on main plot)
# You can hardcode if you want: x_close_thr = 20
x_close_thr = np.quantile(pts["FP/img mean"], 0.60)

# ================== PLOT 1: MAIN (full range) ==================
fig1, ax1 = plt.subplots(figsize=(7.2, 4.6))

# defended points
ax1.scatter(
    pts.loc[pts_is_def, "FP/img mean"],
    pts.loc[pts_is_def, "mAP50-95 (%)"],
    s=45, marker="o", alpha=0.60,
    edgecolors="white", linewidths=0.6,
    label="SVD defended configs", zorder=2
)

# patched baseline (smaller, clean)
ax1.scatter(
    pts.loc[pts_is_patched, "FP/img mean"],
    pts.loc[pts_is_patched, "mAP50-95 (%)"],
    s=50,  # <-- slightly smaller
    marker="D",
    facecolors="white",
    edgecolors="black",
    linewidths=1.2,
    label="PATCHED (no defense)", zorder=5
)

# pareto front (slightly less chunky)
ax1.plot(
    pareto_df["FP/img mean"],
    pareto_df["mAP50-95 (%)"],
    linewidth=2.0, marker="o", markersize=4,  # <-- cleaner
    label="Pareto front", zorder=3
)

# ---- Annotations on MAIN: only far Pareto points + PATCHED ----
pareto_sorted = pareto_df.sort_values("FP/img mean").copy()

# label PATCHED always (move slightly inward so it doesn't hug border)
patched_pareto = pareto_sorted[is_patched_series(pareto_sorted)]
for _, r in patched_pareto.iterrows():
    annotate_clamped(ax1, r["FP/img mean"], r["mAP50-95 (%)"], "PATCHED", dx=8, dy=10, fontsize=10)

# label only SVD pareto points with x > threshold
svd_pareto_far = pareto_sorted[
    is_svd_series(pareto_sorted) & (pareto_sorted["FP/img mean"] > x_close_thr)
].copy()

for _, r in svd_pareto_far.iterrows():
    annotate_clamped(ax1, r["FP/img mean"], r["mAP50-95 (%)"], label_info(r), dx=8, dy=8, fontsize=10)

ax1.set_xlabel(r"False positives per image @ conf=0.001 (↓ better)")
ax1.set_ylabel(r"mAP50-95 (%) @ conf=0.3")

# Clean styling
ax1.grid(True, alpha=0.25, linewidth=0.8)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# padding
x_min, x_max = pts["FP/img mean"].min(), pts["FP/img mean"].max()
y_min, y_max = pts["mAP50-95 (%)"].min(), pts["mAP50-95 (%)"].max()
ax1.set_xlim(x_min - 0.03*(x_max - x_min + 1e-9), x_max + 0.05*(x_max - x_min + 1e-9))
ax1.set_ylim(y_min - 0.10, y_max + 0.15)

ax1.legend(frameon=False, loc="upper left")
fig1.tight_layout()
plt.show()

# ================== PLOT 2: ZOOM (left cluster) ==================
fig2, ax2 = plt.subplots(figsize=(7.2, 4.6))  # match main aspect for consistency

cluster_pts = pts[pts["FP/img mean"] <= x_close_thr].copy()
cluster_pareto = pareto_df[pareto_df["FP/img mean"] <= x_close_thr].sort_values("FP/img mean").copy()

cluster_is_patched = is_patched_series(cluster_pts)
cluster_is_def = ~cluster_is_patched

ax2.scatter(
    cluster_pts.loc[cluster_is_def, "FP/img mean"],
    cluster_pts.loc[cluster_is_def, "mAP50-95 (%)"],
    s=55, marker="o", alpha=0.70,
    edgecolors="white", linewidths=0.6,
    label="SVD defended configs", zorder=2
)

ax2.scatter(
    cluster_pts.loc[cluster_is_patched, "FP/img mean"],
    cluster_pts.loc[cluster_is_patched, "mAP50-95 (%)"],
    s=50,  # <-- smaller here too
    marker="D",
    facecolors="white",
    edgecolors="black",
    linewidths=1.2,
    label="PATCHED (no defense)", zorder=5
)

if len(cluster_pareto) >= 2:
    ax2.plot(
        cluster_pareto["FP/img mean"],
        cluster_pareto["mAP50-95 (%)"],
        linewidth=2.0, marker="o", markersize=4,  # <-- cleaner
        label="Pareto front", zorder=3
    )

# Zoom title (helps readers immediately)
ax2.set_title("Zoomed view of low-FP region")

# annotate ALL SVD pareto points in zoom
svd_zoom = cluster_pareto[is_svd_series(cluster_pareto)].sort_values("mAP50-95 (%)").copy()

# place labels with gentle vertical stacking
for i, (_, r) in enumerate(svd_zoom.iterrows()):
    annotate_clamped(
        ax2,
        r["FP/img mean"], r["mAP50-95 (%)"],
        label_info(r),
        dx=10, dy=8 + 12*i,
        fontsize=9
    )

ax2.set_xlabel(r"False positives per image @ conf=0.001 (↓ better)")
ax2.set_ylabel(r"mAP50-95 (%) @ conf=0.3")

ax2.grid(True, alpha=0.25, linewidth=0.8)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

# tight zoom limits + padding
x2_min, x2_max = cluster_pts["FP/img mean"].min(), cluster_pts["FP/img mean"].max()
y2_min, y2_max = cluster_pts["mAP50-95 (%)"].min(), cluster_pts["mAP50-95 (%)"].max()

ax2.set_xlim(x2_min - 0.08*(x2_max - x2_min + 1e-9), x2_max + 0.12*(x2_max - x2_min + 1e-9))
ax2.set_ylim(y2_min - 0.08, y2_max + 0.10)

ax2.legend(frameon=False, loc="upper left")
fig2.tight_layout()
plt.show()

# Optional saves:
fig1.savefig("pareto_main_fpimg_vs_map.pdf", bbox_inches="tight")
fig2.savefig("pareto_zoom_fpimg_vs_map.pdf", bbox_inches="tight")


In [ ]:
# ===========================
# YOLO EVAL (DEMO-STYLE) on PATCHED vs DEFENDED tensors
# ===========================
import torch
import cv2
import matplotlib.pyplot as plt

# Make sure model exists
if "model" not in globals():
    from ultralytics import YOLO
    MODEL_PATH = "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt"
    model = YOLO(MODEL_PATH)

device = next(model.model.parameters()).device
model.model.eval()

patched_t  = torch.from_numpy(patched).permute(2,0,1).float().to(device) / 255.0
defended_t = torch.from_numpy(defended).permute(2,0,1).float().to(device) / 255.0

with torch.no_grad():
    r_p = model(patched_t.unsqueeze(0),  verbose=False)[0]
    r_d = model(defended_t.unsqueeze(0), verbose=False)[0]

n_p = 0 if r_p.boxes is None else len(r_p.boxes)
n_d = 0 if r_d.boxes is None else len(r_d.boxes)
print(f"Detections (defaults ~conf=0.25): patched={n_p} | defended={n_d}")

vis_p = cv2.cvtColor(r_p.plot(), cv2.COLOR_BGR2RGB)
vis_d = cv2.cvtColor(r_d.plot(), cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14,6))
plt.subplot(1,2,1); plt.imshow(vis_p); plt.title(f"YOLO on PATCHED (n={n_p})"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(vis_d); plt.title(f"YOLO on DEFENDED (n={n_d})"); plt.axis("off")
plt.show()




## Defense 3: Patch Block defense follows a three-stage pipeline (Chunking → Separating → Mitigating)   MITIGATION = STRONG LGS




In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch

# ===========================
# SETTINGS
# ===========================
K = 50
STRIDE = 10
OUTLIER_FRAC = 0.02
BINS = 64

# Strong LGS params (local)
LGS_SIGMA = 5.0
LGS_THRESHOLD = 0.10
MORPH_KERNEL = 7
DILATE_ITERS = 2
MASK_MODE = "laplacian"

# Expand ROI a bit around detected patch (pixels)
ROI_PAD = 20

# ===========================
# sanity checks
# ===========================
for fn in ["chunk_and_extract_features", "separate_anomalous_chunks", "draw_boxes"]:
    if fn not in globals():
        raise NameError(f"Missing `{fn}`. Run your Stage 1/2 definition cells first.")
if "patched" not in globals():
    raise NameError("Missing `patched` (RGB uint8 numpy image). Define it before running this cell.")
if patched.dtype != np.uint8 or patched.ndim != 3 or patched.shape[2] != 3:
    raise ValueError("`patched` must be RGB uint8 with shape (H,W,3).")

H, W = patched.shape[:2]

# ===========================
# Strong LGS (numpy) on an ROI
# ===========================
def lgs_strong_np(img_rgb_uint8, sigma=5.0, threshold=0.10, morph_kernel_size=7, dilate_iters=2, mask_mode="laplacian"):
    img = img_rgb_uint8.astype(np.float32) / 255.0
    gray = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_RGB2GRAY)

    if mask_mode == "laplacian":
        hf = np.abs(cv2.Laplacian(gray, cv2.CV_32F, ksize=3))
    else:
        gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
        hf = cv2.magnitude(gx, gy)

    hf_norm = hf / (hf.max() + 1e-8)
    mask = (hf_norm > threshold).astype(np.uint8)

    if morph_kernel_size and morph_kernel_size > 0:
        k = np.ones((morph_kernel_size, morph_kernel_size), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k)

    if dilate_iters and dilate_iters > 0:
        k2 = np.ones((morph_kernel_size, morph_kernel_size), np.uint8)
        mask = cv2.dilate(mask, k2, iterations=dilate_iters)

    blurred = cv2.GaussianBlur(img, (0,0), sigmaX=sigma, sigmaY=sigma)

    # HARD replace (strong)
    out = img*(1-mask[...,None]) + blurred*mask[...,None]
    out = np.clip(out, 0, 1)
    return (out*255).astype(np.uint8), mask

# ===========================
# Stage 1: Chunking
# ===========================
coords, feats = chunk_and_extract_features(patched, k=K, stride=STRIDE, bins=BINS)

# ===========================
# Stage 2: Separating
# ===========================
keep_idx, scores = separate_anomalous_chunks(coords, feats, (H, W), k=K, stride=STRIDE, outlier_frac=OUTLIER_FRAC)

# Visualize selected windows
boxed = draw_boxes(patched, coords, keep_idx, k=K)
plt.figure(figsize=(7,6))
plt.imshow(boxed); plt.title(f"Patch windows (n={len(keep_idx)})"); plt.axis("off")
plt.show()

# ===========================
# Build ROI bbox from windows
# ===========================
if len(keep_idx) == 0:
    print("No patch windows selected -> no local defense applied.")
    defended = patched.copy()
else:
    ys = [coords[i][0] for i in keep_idx]
    xs = [coords[i][1] for i in keep_idx]

    y1 = max(0, min(ys) - ROI_PAD)
    x1 = max(0, min(xs) - ROI_PAD)
    y2 = min(H, max(ys) + K + ROI_PAD)
    x2 = min(W, max(xs) + K + ROI_PAD)

    # Apply strong LGS only in ROI
    defended = patched.copy()
    roi = defended[y1:y2, x1:x2].copy()
    roi_def, roi_mask = lgs_strong_np(
        roi,
        sigma=LGS_SIGMA,
        threshold=LGS_THRESHOLD,
        morph_kernel_size=MORPH_KERNEL,
        dilate_iters=DILATE_ITERS,
        mask_mode=MASK_MODE
    )
    defended[y1:y2, x1:x2] = roi_def

    # Show ROI mask + defended ROI
    plt.figure(figsize=(14,4))
    plt.subplot(1,3,1); plt.imshow(roi); plt.title("ROI (before)"); plt.axis("off")
    plt.subplot(1,3,2); plt.imshow(roi_mask, cmap="gray"); plt.title("ROI LGS mask"); plt.axis("off")
    plt.subplot(1,3,3); plt.imshow(roi_def); plt.title("ROI (after strong LGS)"); plt.axis("off")
    plt.tight_layout(); plt.show()

    # Show full defended
    vis_full = defended.copy()
    cv2.rectangle(vis_full, (x1,y1), (x2,y2), (255,0,0), 2)
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1); plt.imshow(patched); plt.title("PATCHED"); plt.axis("off")
    plt.subplot(1,2,2); plt.imshow(vis_full); plt.title("DEFENDED (local strong LGS) + ROI box"); plt.axis("off")
    plt.show()

# ===========================
# YOLO eval (demo-style defaults)
# ===========================
device = next(model.model.parameters()).device
model.model.eval()

patched_t  = torch.from_numpy(patched).permute(2,0,1).float().to(device) / 255.0
defended_t = torch.from_numpy(defended).permute(2,0,1).float().to(device) / 255.0

r_p = model(patched_t.unsqueeze(0), verbose=False)[0]
r_d = model(defended_t.unsqueeze(0), verbose=False)[0]

n_p = 0 if r_p.boxes is None else len(r_p.boxes)
n_d = 0 if r_d.boxes is None else len(r_d.boxes)
print(f"Detections (defaults ~conf=0.25): patched={n_p} | defended={n_d}")

vis_p = cv2.cvtColor(r_p.plot(), cv2.COLOR_BGR2RGB)
vis_d = cv2.cvtColor(r_d.plot(), cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14,6))
plt.subplot(1,2,1); plt.imshow(vis_p); plt.title(f"YOLO on PATCHED (n={n_p})"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(vis_d); plt.title(f"YOLO on DEFENDED (n={n_d})"); plt.axis("off")
plt.show()


In [ ]:
# ===========================
# ROI + STRONG LGS SWEEP (5 configs) + FULL EVAL
# - Uses PatchBlock to localize patch windows
# - Builds ROI from windows (+ pad)
# - Builds mask via Laplacian HF + morph + dilation
# - Hard-replaces masked pixels with Gaussian-blurred pixels (strong)
# - Evaluates same metrics as your previous pipeline
# ===========================

!pip -q install ultralytics pyyaml tqdm

import os, glob, shutil, re, yaml
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
import cv2
from ultralytics import YOLO

# ---------------------------
# PATHS
# ---------------------------
MODEL_PATH   = "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt"
PATCHED_ROOT = "/content/drive/MyDrive/Patched Datasets/patched_dataset1"

TMP_BASE = "/content/tmp_roi_lgs_sweep"
os.makedirs(TMP_BASE, exist_ok=True)

# ---------------------------
# PATCHBLOCK (same as your pipeline)
# ---------------------------
K = 50
STRIDE = 10
OUTLIER_FRAC = 0.02
BINS = 64

# ---------------------------
# EVAL params
# ---------------------------
CONFS = [0.001, 0.3, 0.5]
IMGSZ = 640
NMS_IOU = 0.7
IOU_MATCH = 0.50
OVERLOAD_T = 50

# ---------------------------
# 5 STRONG ROI-LGS CONFIGS (paper-friendly ladder)
# name, sigma, thr, morph_kernel, dilate_iters, roi_pad, mask_mode
# ---------------------------
LGS_CONFIGS = [
    ("ROI_LGS_A_balanced",   7.5, 0.08,  7, 2, 25, "laplacian"),
    ("ROI_LGS_B_strong",    10.0, 0.07,  9, 3, 30, "laplacian"),
    ("ROI_LGS_C_stronger",  12.0, 0.06, 11, 3, 35, "laplacian"),
    ("ROI_LGS_D_verystrong",15.0, 0.05, 11, 4, 45, "laplacian"),
    ("ROI_LGS_E_nuclear",   18.0, 0.04, 13, 5, 60, "laplacian"),
]

# ---------------------------
# Sanity: Stage 1/2 must exist
# ---------------------------
for fn in ["chunk_and_extract_features", "separate_anomalous_chunks"]:
    if fn not in globals():
        raise NameError(f"Missing `{fn}`. Run your Stage 1/2 definition cells first.")

def detect_split(root_dir):
    for s in ["test", "val", "valid", "train"]:
        if os.path.exists(os.path.join(root_dir, "images", s)) and os.path.exists(os.path.join(root_dir, "labels", s)):
            return s
    return None

split = detect_split(PATCHED_ROOT)
if split is None:
    raise FileNotFoundError(f"PATCHED_ROOT has no images/{{test|val|valid|train}} and labels/{{...}}:\n{PATCHED_ROOT}")

images_dir = os.path.join(PATCHED_ROOT, "images", split)
labels_dir = os.path.join(PATCHED_ROOT, "labels", split)

img_paths = []
for ext in ("*.png","*.jpg","*.jpeg","*.webp"):
    img_paths += glob.glob(os.path.join(images_dir, ext))
img_paths = sorted(img_paths)
if not img_paths:
    raise RuntimeError(f"No images found in {images_dir}")

print("MODEL:", MODEL_PATH)
print("PATCHED_ROOT:", PATCHED_ROOT)
print("split:", split, "| images:", len(img_paths))

# ---------------------------
# Model
# ---------------------------
model = YOLO(MODEL_PATH)
names = getattr(model.model, "names", None) or getattr(model, "names", None)
if isinstance(names, dict):
    names_list = [names[i] for i in sorted(names.keys())]
elif isinstance(names, list):
    names_list = names
else:
    names_list = None

# ---------------------------
# YAML writer (Ultralytics needs train+val)
# ---------------------------
def write_yaml(root_dir, out_path):
    d = {"path": root_dir, "train": f"images/{split}", "val": f"images/{split}", "test": f"images/{split}"}
    if names_list is not None:
        d["names"] = names_list
    with open(out_path, "w") as f:
        yaml.safe_dump(d, f, sort_keys=False)

# ---------------------------
# Robust label parsing
# ---------------------------
_num_re = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

def yolo_label_to_xyxy(line, w, h):
    nums = _num_re.findall(line)
    if len(nums) < 5:
        return None
    cls = int(float(nums[0]))
    xc, yc, bw, bh = map(float, nums[1:5])
    x1 = (xc - bw/2) * w
    y1 = (yc - bh/2) * h
    x2 = (xc + bw/2) * w
    y2 = (yc + bh/2) * h
    return cls, np.array([x1, y1, x2, y2], dtype=np.float32)

def iou_xyxy(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    iw = max(0.0, x2 - x1)
    ih = max(0.0, y2 - y1)
    inter = iw * ih
    area_a = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    area_b = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])
    return inter / (area_a + area_b - inter + 1e-9)

# ---------------------------
# Strong LGS on ROI (numpy)
# returns (roi_defended_uint8, mask_uint8)
# ---------------------------
def lgs_strong_np(img_rgb_uint8, sigma, threshold, morph_kernel_size, dilate_iters, mask_mode="laplacian"):
    img = img_rgb_uint8.astype(np.float32) / 255.0
    gray = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_RGB2GRAY)

    if mask_mode == "laplacian":
        hf = np.abs(cv2.Laplacian(gray, cv2.CV_32F, ksize=3))
    else:
        gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
        hf = cv2.magnitude(gx, gy)

    hf_norm = hf / (hf.max() + 1e-8)
    mask = (hf_norm > threshold).astype(np.uint8)

    if morph_kernel_size and morph_kernel_size > 0:
        k = np.ones((morph_kernel_size, morph_kernel_size), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k)

    if dilate_iters and dilate_iters > 0:
        k2 = np.ones((morph_kernel_size, morph_kernel_size), np.uint8)
        mask = cv2.dilate(mask, k2, iterations=dilate_iters)

    blurred = cv2.GaussianBlur(img, (0,0), sigmaX=sigma, sigmaY=sigma)

    # HARD replace (strong)
    out = img*(1-mask[...,None]) + blurred*mask[...,None]
    out = np.clip(out, 0, 1)
    return (out*255).astype(np.uint8), mask

# ---------------------------
# Cache PatchBlock localization ONCE
# ---------------------------
print("\nPrecomputing PatchBlock localization once...")
pb_cache = {}  # stem -> {"coords": coords, "keep_idx": keep_idx}

for p in tqdm(img_paths, desc="PatchBlock locate", leave=False):
    stem = os.path.splitext(os.path.basename(p))[0]
    img = np.array(Image.open(p).convert("RGB"), dtype=np.uint8)
    H, W = img.shape[:2]
    coords, feats = chunk_and_extract_features(img, k=K, stride=STRIDE, bins=BINS)
    keep_idx, _ = separate_anomalous_chunks(coords, feats, (H, W), k=K, stride=STRIDE, outlier_frac=OUTLIER_FRAC)
    pb_cache[stem] = {"coords": coords, "keep_idx": keep_idx}

print("✅ Cached:", len(pb_cache), "images")

# ---------------------------
# Build defended dataset for a config
# ALSO logs ROI & mask coverage stats (for debugging / reporting)
# ---------------------------
def build_defended_dataset_roi_lgs(tag, sigma, thr, mk, dil, roi_pad, mode, overwrite=True):
    out_root   = os.path.join(TMP_BASE, tag)
    out_images = os.path.join(out_root, "images", split)
    out_labels = os.path.join(out_root, "labels", split)

    if overwrite and os.path.exists(out_root):
        shutil.rmtree(out_root)

    os.makedirs(out_images, exist_ok=True)
    os.makedirs(out_labels, exist_ok=True)

    # copy labels
    for lab in glob.glob(os.path.join(labels_dir, "*.txt")):
        shutil.copy2(lab, os.path.join(out_labels, os.path.basename(lab)))

    roi_area_fracs = []
    mask_fracs = []

    for p in tqdm(img_paths, desc=f"Defend {tag}", leave=False):
        stem = os.path.splitext(os.path.basename(p))[0]
        img = np.array(Image.open(p).convert("RGB"), dtype=np.uint8)
        H, W = img.shape[:2]

        coords = pb_cache[stem]["coords"]
        keep_idx = pb_cache[stem]["keep_idx"]

        defended = img.copy()

        if len(keep_idx) > 0:
            ys = [coords[i][0] for i in keep_idx]
            xs = [coords[i][1] for i in keep_idx]
            y1 = max(0, min(ys) - roi_pad)
            x1 = max(0, min(xs) - roi_pad)
            y2 = min(H, max(ys) + K + roi_pad)
            x2 = min(W, max(xs) + K + roi_pad)

            roi = defended[y1:y2, x1:x2].copy()
            roi_def, roi_mask = lgs_strong_np(roi, sigma=sigma, threshold=thr,
                                              morph_kernel_size=mk, dilate_iters=dil, mask_mode=mode)
            defended[y1:y2, x1:x2] = roi_def

            roi_area = (y2-y1)*(x2-x1)
            roi_area_fracs.append(roi_area / float(H*W + 1e-9))
            mask_fracs.append(float(roi_mask.mean()))
        else:
            roi_area_fracs.append(0.0)
            mask_fracs.append(0.0)

        Image.fromarray(defended).save(os.path.join(out_images, os.path.basename(p)))

    return out_root, {
        "roi_area_frac_avg": float(np.mean(roi_area_fracs)),
        "mask_frac_avg": float(np.mean(mask_fracs)),
    }

# ---------------------------
# Metrics: FP/Det/TP/FN + overload etc.
# ---------------------------
def compute_counts_metrics(root_dir, conf):
    img_dir = os.path.join(root_dir, "images", split)
    lab_dir = os.path.join(root_dir, "labels", split)

    paths = []
    for ext in ("*.png","*.jpg","*.jpeg","*.webp"):
        paths += glob.glob(os.path.join(img_dir, ext))
    paths = sorted(paths)
    n_imgs = len(paths)

    det_list, fp_list, tp_list, fn_list = [], [], [], []
    overload_count = 0
    total_tp = total_fp = total_fn = 0

    with torch.no_grad():
        for p in paths:
            img = Image.open(p).convert("RGB")
            w, h = img.size
            stem = os.path.splitext(os.path.basename(p))[0]
            lab_path = os.path.join(lab_dir, stem + ".txt")

            gts = []
            if os.path.exists(lab_path):
                with open(lab_path, "r") as f:
                    for line in f:
                        r = yolo_label_to_xyxy(line, w, h)
                        if r is not None:
                            gts.append(r)

            pred = model.predict(source=np.array(img), conf=conf, iou=NMS_IOU, imgsz=IMGSZ, verbose=False)[0]
            boxes = pred.boxes

            if boxes is None or len(boxes) == 0:
                det_count = 0; tp = 0; fp = 0; fn = len(gts)
            else:
                det_xyxy = boxes.xyxy.cpu().numpy()
                det_cls  = boxes.cls.cpu().numpy().astype(int)
                det_count = len(det_xyxy)

                if det_count >= OVERLOAD_T:
                    overload_count += 1

                used_gt = set()
                tp = 0; fp = 0
                for j in range(det_count):
                    pj = det_xyxy[j]; cj = det_cls[j]
                    best_iou, best_k = 0.0, None
                    for k, (cg, gg) in enumerate(gts):
                        if k in used_gt: continue
                        if cg != cj: continue
                        i = iou_xyxy(pj, gg)
                        if i > best_iou:
                            best_iou, best_k = i, k
                    if best_k is not None and best_iou >= IOU_MATCH:
                        used_gt.add(best_k); tp += 1
                    else:
                        fp += 1

                fn = max(0, len(gts) - tp)

            det_list.append(det_count); fp_list.append(fp)
            tp_list.append(tp); fn_list.append(fn)
            total_tp += tp; total_fp += fp; total_fn += fn

    det_arr = np.array(det_list, dtype=float)
    fp_arr  = np.array(fp_list, dtype=float)
    tp_arr  = np.array(tp_list, dtype=float)
    fn_arr  = np.array(fn_list, dtype=float)

    precision = total_tp / (total_tp + total_fp + 1e-9)
    recall    = total_tp / (total_tp + total_fn + 1e-9)

    return {
        "FP/img mean": float(fp_arr.mean()),
        "FP/img std":  float(fp_arr.std(ddof=1)) if n_imgs > 1 else 0.0,
        "Det/img mean": float(det_arr.mean()),
        "Det/img std":  float(det_arr.std(ddof=1)) if n_imgs > 1 else 0.0,
        "TP/img mean": float(tp_arr.mean()),
        "FN/img mean": float(fn_arr.mean()),
        "Precision": float(precision),
        "Recall": float(recall),
        "Overload@50 (%)": float(100.0 * overload_count / max(1, n_imgs)),
        "Det/img median": float(np.median(det_arr)),
        "Det/img p95": float(np.percentile(det_arr, 95)),
        "images": int(n_imgs),
    }

def eval_one_dataset(name, root_dir, conf):
    yaml_path = f"/content/tmp_eval_{re.sub('[^a-zA-Z0-9_]+','_',name)}.yaml"
    write_yaml(root_dir, yaml_path)
    val_res = model.val(data=yaml_path, conf=conf, iou=NMS_IOU, imgsz=IMGSZ, verbose=False)
    map5095 = float(val_res.box.map) * 100.0
    map50   = float(val_res.box.map50) * 100.0
    counts  = compute_counts_metrics(root_dir, conf)
    return {"dataset": name, "conf": conf, "mAP50-95 (%)": map5095, "mAP50 (%)": map50, **counts}

# ---------------------------
# RUN: baseline PATCHED + 5 configs
# ---------------------------
rows = []

print("\n==============================")
print("EVALUATING baseline: PATCHED")
for conf in CONFS:
    r = eval_one_dataset("PATCHED", PATCHED_ROOT, conf)
    rows.append(r)
    print(f"[PATCHED] conf={conf:.3f} | mAP50-95={r['mAP50-95 (%)']:.2f}% | FP/img={r['FP/img mean']:.2f} | Overload={r['Overload@50 (%)']:.1f}%")

for (name, sigma, thr, mk, dil, roi_pad, mode) in LGS_CONFIGS:
    print("\n==============================")
    print("Building defended dataset:", name)

    out_root, dbg = build_defended_dataset_roi_lgs(name, sigma, thr, mk, dil, roi_pad, mode, overwrite=True)
    print(f"ROI avg area fraction: {dbg['roi_area_frac_avg']:.3f} | mask avg coverage in ROI: {dbg['mask_frac_avg']:.3f}")

    print("EVALUATING:", name)
    for conf in CONFS:
        r = eval_one_dataset(name, out_root, conf)
        r["lgs_sigma"] = sigma
        r["lgs_threshold"] = thr
        r["lgs_kernel"] = mk
        r["lgs_dilate"] = dil
        r["roi_pad"] = roi_pad
        r["lgs_mask_mode"] = mode
        r["roi_area_frac_avg"] = dbg["roi_area_frac_avg"]
        r["mask_frac_avg"] = dbg["mask_frac_avg"]
        rows.append(r)

        print(f"[{name}] conf={conf:.3f} | mAP50-95={r['mAP50-95 (%)']:.2f}% | FP/img={r['FP/img mean']:.2f} | Overload={r['Overload@50 (%)']:.1f}%")

df = pd.DataFrame(rows).sort_values(["conf","dataset"]).reset_index(drop=True)

print("\n=== FINAL SUMMARY TABLE ===")
display(df)

out_csv = os.path.join(TMP_BASE, "roi_lgs_sweep_results.csv")
df.to_csv(out_csv, index=False)
print("\n✅ Saved CSV:", out_csv)

# ---------------------------
# Pick "best" for paper:
# minimize overload@0.001, then minimize FP@0.001, then maximize mAP@0.3
# ---------------------------
dS = df[df["conf"] == 0.001][["dataset","FP/img mean","Overload@50 (%)"]].copy()
dA = df[df["conf"] == 0.3][["dataset","mAP50-95 (%)"]].copy().rename(columns={"mAP50-95 (%)":"mAP@0.3"})
rank = dS.merge(dA, on="dataset", how="inner")
rank = rank[rank["dataset"].str.startswith("ROI_LGS_")].copy()

rank = rank.sort_values(["Overload@50 (%)","FP/img mean","mAP@0.3"], ascending=[True, True, False])
print("\n=== PAPER RANKING (best first) ===")
display(rank.head(10))
print("\n✅ BEST:", dict(rank.iloc[0]) if len(rank) else "No ROI_LGS rows found")


In [ ]:
import numpy as np
import pandas as pd

# ------------------ Choose operating points ------------------
CONF_FP  = 0.001   # x-axis
CONF_MAP = 0.3     # y-axis

# If df isn't defined, load your sweep CSV
if "df" not in globals():
    df = pd.read_csv("/content/tmp_roi_lgs_sweep/roi_lgs_sweep_results.csv")

required = ["dataset","conf","mAP50-95 (%)","FP/img mean","Overload@50 (%)"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"df missing columns: {missing}")

# ------------------ Build points table ------------------
d_x = df[df["conf"] == CONF_FP][["dataset","FP/img mean","Overload@50 (%)"]].copy()
d_y = df[df["conf"] == CONF_MAP][["dataset","mAP50-95 (%)"]].copy()

pts = d_x.merge(d_y, on="dataset", how="inner")

# bring LGS params if present (for labeling)
param_cols = ["lgs_sigma","lgs_threshold","lgs_kernel","lgs_dilate","roi_pad","lgs_mask_mode"]
for c in param_cols:
    if c in df.columns:
        tmp = df[df["conf"] == CONF_FP][["dataset", c]].drop_duplicates()
        pts = pts.merge(tmp, on="dataset", how="left")

# ------------------ Ensure CLEAN exists (so you can SEE it) ------------------
# If your ROI-LGS sweep df doesn't include CLEAN, inject it using your earlier CLEAN numbers.
# Edit these if needed.
if not pts["dataset"].astype(str).str.upper().eq("CLEAN").any():
    clean_row = {
        "dataset": "CLEAN",
        "FP/img mean": 24.60,            # CLEAN @ conf=0.001 from your table
        "Overload@50 (%)": 4.0,          # CLEAN @ conf=0.001
        "mAP50-95 (%)": 64.760405,       # CLEAN @ conf=0.3 from your table
    }
    # pad missing param cols with NaN
    for c in param_cols:
        if c in pts.columns and c not in clean_row:
            clean_row[c] = np.nan

    pts = pd.concat([pts, pd.DataFrame([clean_row])], ignore_index=True)

# ------------------ Pareto frontier (PATCHED + defenses only; exclude CLEAN or it dominates everything) ------------------
def pareto_frontier_xy(df_xy, xcol="FP/img mean", ycol="mAP50-95 (%)"):
    d = df_xy.sort_values(xcol).copy()
    best = -1e18
    keep = []
    for _, r in d.iterrows():
        if float(r[ycol]) > best + 1e-9:
            keep.append(True)
            best = float(r[ycol])
        else:
            keep.append(False)
    d["is_pareto"] = keep
    return d

# define which rows are which
def is_clean_series(d):   return d["dataset"].astype(str).str.upper().eq("CLEAN")
def is_patched_series(d): return d["dataset"].astype(str).str.contains("PATCHED", regex=False)

# defenses = everything except CLEAN + PATCHED
is_clean   = is_clean_series(pts)
is_patched = is_patched_series(pts)
is_def     = ~(is_clean | is_patched)

# pareto computed on attacked scenario only (patched + defenses)
pareto_input = pts[~is_clean].copy()
pareto_all   = pareto_frontier_xy(pareto_input)
pareto_df    = pareto_all[pareto_all["is_pareto"]].copy()

# a nice zoom threshold: include defenses + CLEAN, exclude patched
def_fp_max = pts.loc[is_def, "FP/img mean"].max() if is_def.any() else np.nan
clean_fp   = pts.loc[is_clean, "FP/img mean"].max() if is_clean.any() else np.nan
x_close_thr = np.nanmax([def_fp_max + 2, clean_fp + 2, 30])  # robust default

print("pts rows:", len(pts), "| pareto points:", len(pareto_df), "| zoom x_close_thr:", x_close_thr)
display(pts.sort_values("FP/img mean"))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----- choose the two operating points -----
CONF_STRENGTH = 0.001   # x-axis uses FP/img here
CONF_ACC      = 0.3     # y-axis uses mAP here

# ----- keep ONLY ROI_LGS configs + PATCHED baseline (kills CLEAN) -----
df_use = df[
    df["dataset"].astype(str).str.startswith("ROI_LGS") |
    df["dataset"].astype(str).eq("PATCHED")
].copy()

# strength side (FP/img @ conf=0.001)
d_strength = df_use[df_use["conf"] == CONF_STRENGTH][["dataset", "FP/img mean"]].copy()

# accuracy side (mAP @ conf=0.3)
d_acc = df_use[df_use["conf"] == CONF_ACC][["dataset", "mAP50-95 (%)"]].copy()
d_acc = d_acc.rename(columns={"mAP50-95 (%)": "mAP@0.3"})

# merge -> one point per dataset
pts = d_strength.merge(d_acc, on="dataset", how="inner")

# bring params (optional, not plotted)
param_cols = ["lgs_sigma","lgs_threshold","lgs_kernel","lgs_dilate","roi_pad","lgs_mask_mode"]
param_cols = [c for c in param_cols if c in df_use.columns]
if param_cols:
    tmp = df_use[df_use["conf"] == CONF_STRENGTH][["dataset"] + param_cols].drop_duplicates()
    pts = pts.merge(tmp, on="dataset", how="left")

pts


In [ ]:
def pareto_frontier(df_xy, xcol="FP/img mean", ycol="mAP@0.3"):
    d = df_xy.sort_values(xcol).copy()
    best = -1e18
    keep = []
    for _, r in d.iterrows():
        y = float(r[ycol])
        if y > best + 1e-9:
            keep.append(True)
            best = y
        else:
            keep.append(False)
    d["is_pareto"] = keep
    return d[d["is_pareto"]].sort_values(xcol)

pareto_df = pareto_frontier(pts, xcol="FP/img mean", ycol="mAP@0.3")
pareto_df


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----- styling (paper-ish, minimal) -----
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 180,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def_mask     = pts["dataset"].astype(str).str.startswith("ROI_LGS")
patched_mask = pts["dataset"].astype(str).eq("PATCHED")

# ---------------------------
# Helper: annotate Pareto pts (σ, τ only) — ONLY used in ZOOM
# Fixes "dot with no info" by merging params from pts if missing in pareto_df
# ---------------------------
def annotate_pareto_sigma_tau(ax, pareto_zoom, pts_ref, dx=6, dy=6, dy_step=12, fontsize=9):
    needed = ["dataset", "lgs_sigma", "lgs_threshold"]
    # ensure pareto has sigma/threshold; if not, pull them from pts
    for c in ["lgs_sigma", "lgs_threshold"]:
        if c not in pareto_zoom.columns:
            pareto_zoom = pareto_zoom.merge(
                pts_ref[["dataset", c]].drop_duplicates(),
                on="dataset", how="left"
            )

    p = pareto_zoom.sort_values(["FP/img mean", "mAP@0.3"]).copy()

    for i, (_, r) in enumerate(p.iterrows()):
        sig = r.get("lgs_sigma", np.nan)
        thr = r.get("lgs_threshold", np.nan)

        # ALWAYS put a label (so no unlabeled dot):
        if pd.isna(sig) or pd.isna(thr):
            txt = str(r["dataset"])  # fallback if params missing
        else:
            txt = f"σ={float(sig):g}, τ={float(thr):g}"

        ax.annotate(
            txt,
            (float(r["FP/img mean"]), float(r["mAP@0.3"])),
            textcoords="offset points",
            xytext=(dx, dy + i * dy_step),
            ha="left", va="bottom",
            fontsize=fontsize,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.85),
            arrowprops=dict(arrowstyle="-", linewidth=0.8, alpha=0.55),
            zorder=10,
        )

# =======================
# FIGURE A: MAIN (NO parameter annotations here)
# =======================
fig, ax = plt.subplots(figsize=(7.2, 4.6))

ax.scatter(
    pts.loc[def_mask, "FP/img mean"],
    pts.loc[def_mask, "mAP@0.3"],
    s=55, alpha=0.75,
    edgecolors="white", linewidths=0.7,
    label="ROI-LGS defended configs",
    zorder=2,
)

ax.plot(
    pareto_df["FP/img mean"],
    pareto_df["mAP@0.3"],
    lw=2.2, marker="o", markersize=4,
    label="Pareto front",
    zorder=3,
)

if patched_mask.any():
    ax.scatter(
        pts.loc[patched_mask, "FP/img mean"],
        pts.loc[patched_mask, "mAP@0.3"],
        s=90, marker="D",
        facecolors="white", edgecolors="black", linewidths=1.4,
        label="PATCHED (no defense)",
        zorder=5,
    )

ax.set_title("ROI-LGS: strength–accuracy trade-off")
ax.set_xlabel(r"False positives per image @ conf=0.001 (↓ better)")
ax.set_ylabel(r"mAP50-95 (%) @ conf=0.3 (↑ better)")

ax.grid(True, alpha=0.18, linewidth=0.8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

x = pts["FP/img mean"].astype(float).values
y = pts["mAP@0.3"].astype(float).values
ax.set_xlim(x.min() - 0.08*(x.max()-x.min()+1e-9), x.max() + 0.10*(x.max()-x.min()+1e-9))
ax.set_ylim(y.min() - 1.0, y.max() + 1.0)

ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
plt.show()

fig.savefig("pareto_roi_lgs_main.pdf", bbox_inches="tight")
fig.savefig("pareto_roi_lgs_main.png", bbox_inches="tight")




In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----- styling (paper-ish, minimal) -----
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 180,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

COLOR = "tab:blue"  # ✅ same as your reference plot

def_mask = pts["dataset"].astype(str).str.startswith("ROI_LGS")

# ---------------------------
# Helper: label ALL dots with σ,τ (fallback dataset name if missing)
# Also pulls sigma/threshold from pts if missing in the plotting df
# ---------------------------
def label_all_sigma_tau(ax, df_points, pts_ref, xcol="FP/img mean", ycol="mAP@0.3",
                        dx=10, base_dy=10, step_dy=14, fontsize=10, x_eps=0.12):

    d = df_points.copy()

    # ensure sigma/threshold exist (merge from pts if missing)
    for c in ["lgs_sigma", "lgs_threshold"]:
        if c not in d.columns:
            d = d.merge(
                pts_ref[["dataset", c]].drop_duplicates(),
                on="dataset", how="left"
            )

    d = d.sort_values([xcol, ycol]).reset_index(drop=True)

    last_x = None
    stack_i = 0

    for _, r in d.iterrows():
        x = float(r[xcol])
        y = float(r[ycol])

        sig = r.get("lgs_sigma", np.nan)
        thr = r.get("lgs_threshold", np.nan)

        if pd.isna(sig) or pd.isna(thr):
            txt = str(r["dataset"])  # fallback if params missing
        else:
            txt = f"σ={float(sig):g}, τ={float(thr):g}"

        # stack labels when x is very close (reduces overlap)
        if last_x is None or abs(x - last_x) > x_eps:
            stack_i = 0
            last_x = x
        else:
            stack_i += 1

        ax.annotate(
            txt, (x, y),
            textcoords="offset points",
            xytext=(dx, base_dy + step_dy * stack_i),
            ha="left", va="bottom",
            fontsize=fontsize,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.85),
            arrowprops=dict(arrowstyle="-", linewidth=0.8, alpha=0.55),
            zorder=10,
        )

# =======================
# FIGURE: ZOOM (defenses only) + Pareto + labels on ALL dots
# =======================
fig, ax = plt.subplots(figsize=(7.2, 4.6))

pts_zoom = pts[def_mask].copy()

pareto_zoom = pareto_df[pareto_df["dataset"].astype(str).str.startswith("ROI_LGS")].copy()
pareto_zoom = pareto_zoom.sort_values("FP/img mean").reset_index(drop=True)

# All defended dots
ax.scatter(
    pts_zoom["FP/img mean"], pts_zoom["mAP@0.3"],
    s=60, alpha=0.75,
    color=COLOR,                # ✅ force same color
    edgecolors="white", linewidths=0.7,
    label="ROI-LGS defended configs",
    zorder=2
)

# Pareto front (same color)
ax.plot(
    pareto_zoom["FP/img mean"],
    pareto_zoom["mAP@0.3"],
    lw=2.2, marker="o", markersize=4,
    color=COLOR,                # ✅ same color
    label="Pareto front",
    zorder=4
)

# Label ALL dots (not only Pareto) with σ,τ
label_all_sigma_tau(
    ax,
    df_points=pts_zoom,
    pts_ref=pts,
    xcol="FP/img mean",
    ycol="mAP@0.3",
    dx=10,
    base_dy=10,
    step_dy=14,
    fontsize=10,
    x_eps=0.12
)

ax.set_title("Zoom: defended region")
ax.set_xlabel(r"False positives per image @ conf=0.001 (↓ better)")
ax.set_ylabel(r"mAP50-95 (%) @ conf=0.3 (↑ better)")

ax.grid(True, alpha=0.18, linewidth=0.8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# tight zoom limits + padding
xv = pts_zoom["FP/img mean"].astype(float).values
yv = pts_zoom["mAP@0.3"].astype(float).values
ax.set_xlim(xv.min() - 0.12*(xv.max()-xv.min()+1e-9), xv.max() + 0.18*(xv.max()-xv.min()+1e-9))
ax.set_ylim(yv.min() - 1.0, yv.max() + 1.0)

ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
plt.show()

fig.savefig("pareto_roi_lgs_zoom.pdf", bbox_inches="tight")
fig.savefig("pareto_roi_lgs_zoom.png", bbox_inches="tight")


In [ ]:
import numpy as np

DATASET_ROOT = "/content/drive/MyDrive/Patched Datasets/patched_dataset1"
OUT_ROOT     = "/content/drive/MyDrive/Patched Datasets/Defended/patched_dataset1_defended_PatchBlock_StrongLGS"

# PatchBlock params
K = 50
STRIDE = 10
OUTLIER_FRAC = 0.02
BINS = 64

#BEST ROI-LGS params (ROI_LGS_D_verystrong)
ROI_PAD       = 45
LGS_SIGMA     = 15.0
LGS_THRESHOLD = 0.05
MORPH_KERNEL  = 11
DILATE_ITERS  = 4
MASK_MODE     = "laplacian"


def defend_patchblock_roi_strong_lgs(img_rgb_uint8: np.ndarray) -> np.ndarray:
    H, W = img_rgb_uint8.shape[:2]
    coords, feats = chunk_and_extract_features(img_rgb_uint8, k=K, stride=STRIDE, bins=BINS)
    keep_idx, _ = separate_anomalous_chunks(
        coords, feats, (H, W),
        k=K, stride=STRIDE, outlier_frac=OUTLIER_FRAC
    )

    if len(keep_idx) == 0:
        return img_rgb_uint8  # or img_rgb_uint8.copy()

    ys = [coords[i][0] for i in keep_idx]
    xs = [coords[i][1] for i in keep_idx]

    y1 = max(0, min(ys) - ROI_PAD)
    x1 = max(0, min(xs) - ROI_PAD)
    y2 = min(H, max(ys) + K + ROI_PAD)
    x2 = min(W, max(xs) + K + ROI_PAD)

    out = img_rgb_uint8.copy()
    roi = out[y1:y2, x1:x2].copy()

    roi_def, _ = lgs_strong_np(   # <-- THIS is the fix
        roi,
        sigma=LGS_SIGMA,
        threshold=LGS_THRESHOLD,
        morph_kernel_size=MORPH_KERNEL,
        dilate_iters=DILATE_ITERS,
        mask_mode=MASK_MODE
    )

    out[y1:y2, x1:x2] = roi_def
    return out


build_defended_dataset(DATASET_ROOT, OUT_ROOT, defend_patchblock_roi_strong_lgs, overwrite=False)
print("Saved:", OUT_ROOT)


## DEFENSE EVALUATION

In [ ]:
!pip -q install ultralytics pyyaml tqdm

import os, glob, re, yaml
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
from ultralytics import YOLO

# ---------------------------
# SETTINGS (EDIT THESE)
# ---------------------------
MODEL_PATH = "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt"

# Put your 3 dataset roots here (each must have images/test and labels/test)
DATASETS = [
    ("CLEAN",   "/content/drive/MyDrive/Patched Datasets/Clean/clean_dataset1"),
    ("PATCHED", "/content/drive/MyDrive/Patched Datasets/patched_dataset1"),
    ("DEFENDED","/content/drive/MyDrive/Patched Datasets/Defended/patched_dataset1_defended_PatchBlock_StrongLGS"),
]

CONFS = [0.001, 0.3, 0.5]       # edit as you like
IMGSZ = 640
NMS_IOU = 0.7              # NMS IoU used for predict + val
IOU_MATCH = 0.50           # IoU threshold for TP matching (FP/img metric)
OVERLOAD_K = 50            # overload if det>=K
MAX_DET = 300              # keep stable

# ---------------------------
# Load Model
# ---------------------------
model = YOLO(MODEL_PATH)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Model:", MODEL_PATH)
print("Device:", device)

# optional names for YAML
names = getattr(model.model, "names", None) or getattr(model, "names", None)
if isinstance(names, dict):
    names_list = [names[i] for i in sorted(names.keys())]
elif isinstance(names, list):
    names_list = names
else:
    names_list = None


In [ ]:
_num_re = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

def write_eval_yaml(root_dir, out_path):
    # Ultralytics requires train+val keys; point them to test for evaluation
    d = {"path": root_dir, "train": "images/test", "val": "images/test", "test": "images/test"}
    if names_list is not None:
        d["names"] = names_list
    with open(out_path, "w") as f:
        yaml.safe_dump(d, f, sort_keys=False)

def list_images(images_dir):
    exts = ("*.png","*.jpg","*.jpeg","*.webp")
    paths = []
    for e in exts:
        paths.extend(glob.glob(os.path.join(images_dir, e)))
    return sorted(paths)

def yolo_label_to_xyxy(line, w, h):
    # robust: first 5 numeric tokens -> cls xc yc bw bh
    nums = _num_re.findall(line)
    if len(nums) < 5:
        return None
    cls = int(float(nums[0]))
    xc, yc, bw, bh = map(float, nums[1:5])
    x1 = (xc - bw/2) * w
    y1 = (yc - bh/2) * h
    x2 = (xc + bw/2) * w
    y2 = (yc + bh/2) * h
    return cls, np.array([x1, y1, x2, y2], dtype=np.float32)

def iou_xyxy(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter = inter_w * inter_h
    area_a = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    area_b = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])
    union = area_a + area_b - inter + 1e-9
    return inter / union

def per_image_stats(images_dir, labels_dir, conf):
    img_paths = list_images(images_dir)
    if not img_paths:
        raise RuntimeError(f"No images found in {images_dir}")

    fp_list, det_list, tp_list, fn_list = [], [], [], []

    with torch.no_grad():
        for p in tqdm(img_paths, desc=f"Per-image stats @ conf={conf}", leave=False):
            img = Image.open(p).convert("RGB")
            w, h = img.size

            stem = os.path.splitext(os.path.basename(p))[0]
            lab_path = os.path.join(labels_dir, stem + ".txt")

            gts = []
            if os.path.exists(lab_path):
                with open(lab_path, "r") as f:
                    for line in f:
                        r = yolo_label_to_xyxy(line, w, h)
                        if r is not None:
                            gts.append(r)

            pred = model.predict(
                source=np.array(img),
                conf=conf,
                iou=NMS_IOU,
                imgsz=IMGSZ,
                max_det=MAX_DET,
                verbose=False
            )[0]

            boxes = pred.boxes
            if boxes is None or len(boxes) == 0:
                fp_list.append(0)
                det_list.append(0)
                tp_list.append(0)
                fn_list.append(len(gts))
                continue

            det_xyxy = boxes.xyxy.cpu().numpy()
            det_cls  = boxes.cls.cpu().numpy().astype(int)

            det_count = len(det_xyxy)
            used_gt = set()
            tp_count = 0
            fp_count = 0

            # greedy, class-aware matching preds->GT
            for j in range(det_count):
                pj = det_xyxy[j]
                cj = det_cls[j]

                best_iou, best_k = 0.0, None
                for k, (cg, gg) in enumerate(gts):
                    if k in used_gt:
                        continue
                    if cg != cj:
                        continue
                    i = iou_xyxy(pj, gg)
                    if i > best_iou:
                        best_iou, best_k = i, k

                if best_k is not None and best_iou >= IOU_MATCH:
                    used_gt.add(best_k)
                    tp_count += 1
                else:
                    fp_count += 1

            fn_count = max(0, len(gts) - tp_count)

            fp_list.append(fp_count)
            det_list.append(det_count)
            tp_list.append(tp_count)
            fn_list.append(fn_count)

    return (np.array(fp_list, dtype=float),
            np.array(det_list, dtype=float),
            np.array(tp_list, dtype=float),
            np.array(fn_list, dtype=float),
            len(img_paths))


In [ ]:
def eval_dataset(name, root_dir):
    images_dir = os.path.join(root_dir, "images", "test")
    labels_dir = os.path.join(root_dir, "labels", "test")

    if not os.path.exists(images_dir):
        raise FileNotFoundError(f"[{name}] Missing: {images_dir}")
    if not os.path.exists(labels_dir):
        raise FileNotFoundError(f"[{name}] Missing: {labels_dir}")

    yaml_path = f"/content/tmp_eval_{name}.yaml"
    write_eval_yaml(root_dir, yaml_path)

    rows = []
    for conf in CONFS:
        # --- mAP ---
        val_res = model.val(data=yaml_path, conf=conf, iou=NMS_IOU, imgsz=IMGSZ, verbose=False)
        map5095 = float(val_res.box.map)
        map50   = float(val_res.box.map50)

        # --- per-image FP/Det/TP/FN ---
        fp, det, tp, fn, n_imgs = per_image_stats(images_dir, labels_dir, conf)

        fp_mean  = float(fp.mean());  fp_std  = float(fp.std(ddof=1)) if n_imgs > 1 else 0.0
        det_mean = float(det.mean()); det_std = float(det.std(ddof=1)) if n_imgs > 1 else 0.0
        tp_mean  = float(tp.mean());  fn_mean = float(fn.mean())

        # totals for precision/recall (micro)
        TP = float(tp.sum())
        FP = float(fp.sum())
        FN = float(fn.sum())
        precision = TP / (TP + FP + 1e-9)
        recall    = TP / (TP + FN + 1e-9)

        overload_pct = float((det >= OVERLOAD_K).mean() * 100.0)
        det_median = float(np.median(det))
        det_p95 = float(np.percentile(det, 95))

        rows.append({
            "dataset": name,
            "conf": conf,
            "mAP50-95 (%)": map5095 * 100,
            "mAP50 (%)": map50 * 100,
            "FP/img mean": fp_mean,
            "FP/img std": fp_std,
            "Det/img mean": det_mean,
            "Det/img std": det_std,
            "TP/img mean": tp_mean,
            "FN/img mean": fn_mean,
            "Precision": precision,
            "Recall": recall,
            f"Overload@{OVERLOAD_K} (%)": overload_pct,
            "Det/img median": det_median,
            "Det/img p95": det_p95,
            "images": n_imgs
        })

        print(
            f"[{name}] conf={conf:.3f} | "
            f"mAP50-95={map5095*100:.2f}% mAP50={map50*100:.2f}% | "
            f"FP/img={fp_mean:.3f}±{fp_std:.3f} Det/img={det_mean:.3f}±{det_std:.3f} | "
            f"P={precision:.3f} R={recall:.3f} | Overload@{OVERLOAD_K}={overload_pct:.1f}%"
        )

    return pd.DataFrame(rows)


In [ ]:
all_df = []
for name, root in DATASETS:
    print("\n==============================")
    print("EVALUATING:", name)
    print("ROOT:", root)
    all_df.append(eval_dataset(name, root))

df = pd.concat(all_df, ignore_index=True)
df = df.sort_values(["conf", "dataset"])

print("\n=== FINAL SUMMARY TABLE ===")
display(df)

OUT_CSV = "/content/drive/MyDrive/Patched Datasets/Defended/eval_3datasets_summary.csv"
df.to_csv(OUT_CSV, index=False)
print("\n✅ Saved CSV:", OUT_CSV)


In [ ]:
# ===========================
# PAPER-CONSISTENT EVAL (ALL DATASETS)
# - Ultralytics metrics: P/R/mAP
# - Perception-load metrics (Det/img, FP/img, Overload@50) are CLASS-AGNOSTIC (paper definition)
# - Pinned ultralytics==8.4.7, NMS_IOU=0.6 (paper)
# ===========================

!pip -q install ultralytics==8.4.7 pyyaml tqdm

import os, glob, re, yaml
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
import ultralytics
from ultralytics import YOLO

print("Ultralytics version:", ultralytics.__version__)

# ---------------------------
# EDIT THESE
# ---------------------------
MODEL_PATH = "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt"

DATASETS = [
    ("PATCHED",          "/content/drive/MyDrive/Patched Datasets/patched_dataset1"),
    ("DEFENDED_LGS",     "/content/drive/MyDrive/Patched Datasets/Defended/patched_dataset1_defended_LGS"),
    ("DEFENDED_PB_SVD",  "/content/drive/MyDrive/Patched Datasets/Defended/patched_dataset1_defended_PatchBlock_SVD"),
    ("DEFENDED_PB_LGS",  "/content/drive/MyDrive/Patched Datasets/Defended/patched_dataset1_defended_PatchBlock_StrongLGS"),
    ("CLEAN",            "/content/drive/MyDrive/Patched Datasets/Clean/clean_dataset1"),
]

# ---------------------------
# PAPER SETTINGS
# ---------------------------
TAUS      = [0.001, 0.3, 0.5]  # you care about >=0.3, but paper also uses 0.001 for load
IMGSZ     = 640
NMS_IOU   = 0.6               # PAPER (your notebook used 0.7)
IOU_MATCH = 0.50              # paper uses IoU>=0.5 for matching
OVERLOAD_K = 50               # Overload@50 in paper
TAU_LOAD   = 0.001            # overload computed at tau_load

EXTS = ("*.png", "*.jpg", "*.jpeg", "*.webp")  # keep broad; won’t hurt if you don’t have webp

# ---------------------------
# Model
# ---------------------------
model = YOLO(MODEL_PATH)

# Optional names for yaml
names = getattr(model.model, "names", None) or getattr(model, "names", None)
if isinstance(names, dict):
    names_list = [names[i] for i in sorted(names.keys())]
elif isinstance(names, list):
    names_list = names
else:
    names_list = None

# ---------------------------
# Helpers
# ---------------------------
def detect_split(root_dir):
    for s in ["test", "val", "valid", "train"]:
        if os.path.exists(os.path.join(root_dir, "images", s)) and os.path.exists(os.path.join(root_dir, "labels", s)):
            return s
    return None

def write_yaml(root_dir, split, out_path):
    d = {"path": root_dir, "train": f"images/{split}", "val": f"images/{split}", "test": f"images/{split}"}
    if names_list is not None:
        d["names"] = names_list
    with open(out_path, "w") as f:
        yaml.safe_dump(d, f, sort_keys=False)

_num_re = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

def yolo_label_to_xyxy(line, w, h):
    nums = _num_re.findall(line)
    if len(nums) < 5:
        return None
    cls = int(float(nums[0]))
    xc, yc, bw, bh = map(float, nums[1:5])
    x1 = (xc - bw/2) * w
    y1 = (yc - bh/2) * h
    x2 = (xc + bw/2) * w
    y2 = (yc + bh/2) * h
    return cls, np.array([x1, y1, x2, y2], dtype=np.float32)

def iou_xyxy(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    iw = max(0.0, x2 - x1)
    ih = max(0.0, y2 - y1)
    inter = iw * ih
    area_a = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    area_b = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])
    return inter / (area_a + area_b - inter + 1e-9)

def list_images(images_dir):
    paths = []
    for e in EXTS:
        paths.extend(glob.glob(os.path.join(images_dir, e)))
    return sorted(paths)

@torch.no_grad()
def load_metrics_class_agnostic(images_dir, labels_dir, tau):
    """
    PAPER perception-load metric:
    FP = prediction with NO overlap (IoU>=0.5) with ANY GT, regardless of class.
    Det = number of predictions >= tau (post-NMS).
    """
    img_paths = list_images(images_dir)
    n = len(img_paths)
    if n == 0:
        raise RuntimeError(f"No images in {images_dir}")

    det_list, fp_list = [], []

    for p in tqdm(img_paths, desc=f"Load metrics @tau={tau}", leave=False):
        img = Image.open(p).convert("RGB")
        w, h = img.size
        stem = os.path.splitext(os.path.basename(p))[0]
        lab_path = os.path.join(labels_dir, stem + ".txt")

        # load GT boxes (ignore class for load metrics)
        gt_boxes = []
        if os.path.exists(lab_path):
            with open(lab_path, "r") as f:
                for line in f:
                    r = yolo_label_to_xyxy(line, w, h)
                    if r is not None:
                        gt_boxes.append(r[1])  # ONLY box

        pred = model.predict(
            source=np.array(img),
            conf=tau,
            iou=NMS_IOU,
            imgsz=IMGSZ,
            verbose=False,
            max_det=300
        )[0]

        boxes = pred.boxes
        if boxes is None or len(boxes) == 0:
            det_list.append(0)
            fp_list.append(0)
            continue

        det_xyxy = boxes.xyxy.cpu().numpy()
        det_count = len(det_xyxy)

        fp = 0
        for j in range(det_count):
            pj = det_xyxy[j]
            explained = False
            for gg in gt_boxes:
                if iou_xyxy(pj, gg) >= IOU_MATCH:
                    explained = True
                    break
            if not explained:
                fp += 1

        det_list.append(det_count)
        fp_list.append(fp)

    det_arr = np.array(det_list, dtype=float)
    fp_arr  = np.array(fp_list, dtype=float)

    return {
        "Det/img mean": float(det_arr.mean()),
        "Det/img std":  float(det_arr.std(ddof=1)) if n > 1 else 0.0,
        "FP/img mean":  float(fp_arr.mean()),
        "FP/img std":   float(fp_arr.std(ddof=1)) if n > 1 else 0.0,
        "images": int(n),
    }, det_arr  # return det_arr so we can compute overload cleanly

def eval_dataset_all_taus(dataset_name, root_dir):
    split = detect_split(root_dir)
    if split is None:
        raise FileNotFoundError(f"[{dataset_name}] Missing images/<split> and labels/<split> under: {root_dir}")

    images_dir = os.path.join(root_dir, "images", split)
    labels_dir = os.path.join(root_dir, "labels", split)

    # overload computed ONCE at tau_load
    load0, det_arr0 = load_metrics_class_agnostic(images_dir, labels_dir, TAU_LOAD)
    overload_pct = float(100.0 * np.mean(det_arr0 >= OVERLOAD_K))

    rows = []
    for tau in TAUS:
        # Ultralytics metrics (P/R/mAP)
        yaml_path = f"/content/tmp_eval_{re.sub('[^a-zA-Z0-9_]+','_',dataset_name)}_{split}.yaml"
        write_yaml(root_dir, split, yaml_path)

        val_res = model.val(data=yaml_path, conf=tau, iou=NMS_IOU, imgsz=IMGSZ, verbose=False)

        P = float(val_res.box.mp) * 100.0
        R = float(val_res.box.mr) * 100.0
        mAP50 = float(val_res.box.map50) * 100.0
        mAP5095 = float(val_res.box.map) * 100.0

        load, _ = load_metrics_class_agnostic(images_dir, labels_dir, tau)

        rows.append({
            "dataset": dataset_name,
            "conf": tau,
            "images": load["images"],
            "P (%)": P,
            "R (%)": R,
            "mAP50 (%)": mAP50,
            "mAP50-95 (%)": mAP5095,
            "Det/img mean": load["Det/img mean"],
            "Det/img std":  load["Det/img std"],
            "FP/img mean":  load["FP/img mean"],
            "FP/img std":   load["FP/img std"],
            f"Overload@{OVERLOAD_K}@{TAU_LOAD} (%)": overload_pct,
        })

    return rows

# ---------------------------
# RUN
# ---------------------------
all_rows = []
for name, root in DATASETS:
    print("\n==============================")
    print("DATASET:", name)
    print("ROOT:", root)
    all_rows.extend(eval_dataset_all_taus(name, root))

df = pd.DataFrame(all_rows).sort_values(["conf","dataset"]).reset_index(drop=True)
display(df)

out_csv = "/content/defended_eval_paper_consistent.csv"
df.to_csv(out_csv, index=False)
print("\n✅ Saved CSV:", out_csv)

In [ ]:
# ===========================
# PAPER-CONSISTENT EVAL (ALL DATASETS)
# - Ultralytics val for P/R/mAP (conf_min=0.001 so mAP is "full")
# - Paper perception-load metrics:
#   * tau_load = 0.001 for Overload@50
#   * FP/img@tau is CLASS-AGNOSTIC "hallucinations" (no IoU>=0.5 overlap with ANY GT)
#   * Det/img@tau counts total detections above tau
# - Also reports tau in {0.3, 0.5} (what you care about)
# ===========================

!pip -q install ultralytics pyyaml tqdm

import os, glob, re, yaml
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
from ultralytics import YOLO

# ---------------------------
# EDIT THESE
# ---------------------------
MODEL_PATH = "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt"

DATASETS = [
    ("PATCHED",        "/content/drive/MyDrive/Patched Datasets/patched_dataset1"),
    ("DEFENDED_LGS",   "/content/drive/MyDrive/Patched Datasets/Defended/patched_dataset1_defended_LGS"),
    ("DEFENDED_PB_SVD","/content/drive/MyDrive/Patched Datasets/Defended/patched_dataset1_defended_PatchBlock_SVD"),
    ("DEFENDED_PB_LGS","/content/drive/MyDrive/Patched Datasets/Defended/patched_dataset1_defended_PatchBlock_StrongLGS"),
    ("CLEAN",          "/content/drive/MyDrive/Patched Datasets/Clean/clean_dataset1"),
]

# ---------------------------
# PAPER SETTINGS
# ---------------------------
IMGSZ      = 640
NMS_IOU    = 0.6      # paper uses 0.6
IOU_MATCH  = 0.50     # paper uses IoU>=0.5 overlap to be "explained"
TAU_LOAD   = 0.001    # paper tau_load
TAUS       = [0.3, 0.5]  # you care about these
OVERLOAD_K = 50

EXTS = ("*.png", "*.jpg", "*.jpeg", "*.webp")

# ---------------------------
# Model
# ---------------------------
model = YOLO(MODEL_PATH)

# optional names for yaml
names = getattr(model.model, "names", None) or getattr(model, "names", None)
if isinstance(names, dict):
    names_list = [names[i] for i in sorted(names.keys())]
elif isinstance(names, list):
    names_list = names
else:
    names_list = None

# ---------------------------
# Helpers
# ---------------------------
def detect_split(root_dir):
    for s in ["test", "val", "valid", "train"]:
        if os.path.exists(os.path.join(root_dir, "images", s)) and os.path.exists(os.path.join(root_dir, "labels", s)):
            return s
    return None

def write_yaml(root_dir, split, out_path):
    d = {"path": root_dir, "train": f"images/{split}", "val": f"images/{split}", "test": f"images/{split}"}
    if names_list is not None:
        d["names"] = names_list
    with open(out_path, "w") as f:
        yaml.safe_dump(d, f, sort_keys=False)

_num_re = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

def yolo_label_to_xyxy(line, w, h):
    nums = _num_re.findall(line)
    if len(nums) < 5:
        return None
    cls = int(float(nums[0]))
    xc, yc, bw, bh = map(float, nums[1:5])
    x1 = (xc - bw/2) * w
    y1 = (yc - bh/2) * h
    x2 = (xc + bw/2) * w
    y2 = (yc + bh/2) * h
    return cls, np.array([x1, y1, x2, y2], dtype=np.float32)

def iou_one_to_many(box, gts):
    # box: (4,), gts: (M,4)
    if gts.size == 0:
        return np.array([], dtype=np.float32)
    x1 = np.maximum(box[0], gts[:,0]); y1 = np.maximum(box[1], gts[:,1])
    x2 = np.minimum(box[2], gts[:,2]); y2 = np.minimum(box[3], gts[:,3])
    iw = np.maximum(0.0, x2 - x1)
    ih = np.maximum(0.0, y2 - y1)
    inter = iw * ih
    area_b = np.maximum(0.0, box[2]-box[0]) * np.maximum(0.0, box[3]-box[1])
    area_g = np.maximum(0.0, gts[:,2]-gts[:,0]) * np.maximum(0.0, gts[:,3]-gts[:,1])
    return inter / (area_b + area_g - inter + 1e-9)

def list_images(images_dir):
    paths = []
    for e in EXTS:
        paths += glob.glob(os.path.join(images_dir, e))
    return sorted(paths)

def load_gt_map(labels_dir, image_paths):
    # Map: stem -> np.array of GT boxes (M,4). Class is ignored for paper FP metrics.
    gt_map = {}
    for p in image_paths:
        stem = os.path.splitext(os.path.basename(p))[0]
        lab_path = os.path.join(labels_dir, stem + ".txt")
        gt_boxes = []
        if os.path.exists(lab_path):
            # read w,h from image for correct de-normalization
            img = Image.open(p).convert("RGB")
            w, h = img.size
            with open(lab_path, "r") as f:
                for line in f:
                    r = yolo_label_to_xyxy(line, w, h)
                    if r is not None:
                        _, xyxy = r
                        gt_boxes.append(xyxy)
        gt_map[stem] = np.stack(gt_boxes, axis=0) if len(gt_boxes) else np.zeros((0,4), dtype=np.float32)
    return gt_map

@torch.no_grad()
def det_fp_overload(images_dir, labels_dir, tau, gt_map=None):
    """
    Paper-style:
      Det/img@tau = mean number of predicted boxes with conf >= tau
      FP/img@tau  = mean number of predictions that do NOT overlap ANY GT (IoU>=0.5), class-agnostic
    """
    img_paths = list_images(images_dir)
    if len(img_paths) == 0:
        raise RuntimeError(f"No images found in {images_dir}")

    if gt_map is None:
        gt_map = load_gt_map(labels_dir, img_paths)

    det_counts = []
    fp_counts  = []

    for p in tqdm(img_paths, desc=f"predict @ tau={tau}", leave=False):
        stem = os.path.splitext(os.path.basename(p))[0]
        gts = gt_map.get(stem, np.zeros((0,4), dtype=np.float32))

        img = Image.open(p).convert("RGB")
        pred = model.predict(source=np.array(img), conf=tau, iou=NMS_IOU, imgsz=IMGSZ, verbose=False)[0]
        boxes = pred.boxes

        if boxes is None or len(boxes) == 0:
            det_counts.append(0)
            fp_counts.append(0)
            continue

        det_xyxy = boxes.xyxy.cpu().numpy()
        det_counts.append(len(det_xyxy))

        # CLASS-AGNOSTIC "explained if overlaps ANY GT with IoU>=0.5"
        fp = 0
        for b in det_xyxy:
            ious = iou_one_to_many(b, gts)
            explained = (ious.size > 0) and (ious.max() >= IOU_MATCH)
            if not explained:
                fp += 1
        fp_counts.append(fp)

    det_arr = np.array(det_counts, dtype=float)
    fp_arr  = np.array(fp_counts, dtype=float)

    return {
        "images": int(len(img_paths)),
        "Det/img mean": float(det_arr.mean()),
        "Det/img std":  float(det_arr.std(ddof=1)) if len(img_paths) > 1 else 0.0,
        "FP/img mean":  float(fp_arr.mean()),
        "FP/img std":   float(fp_arr.std(ddof=1)) if len(img_paths) > 1 else 0.0,
        "Total det":    int(det_arr.sum()),
        "Total FP":     int(fp_arr.sum()),
    }, gt_map

def eval_dataset(name, root_dir):
    split = detect_split(root_dir)
    if split is None:
        raise FileNotFoundError(f"[{name}] Missing images/<split> and labels/<split> under: {root_dir}")

    images_dir = os.path.join(root_dir, "images", split)
    labels_dir = os.path.join(root_dir, "labels", split)

    # ---- Ultralytics detection quality metrics (paper: full eval set, conf_min=0.001) ----
    yaml_path = f"/content/tmp_{re.sub('[^a-zA-Z0-9_]+','_',name)}_{split}.yaml"
    write_yaml(root_dir, split, yaml_path)
    val_res = model.val(data=yaml_path, conf=TAU_LOAD, iou=NMS_IOU, imgsz=IMGSZ, verbose=False)

    # Ultralytics returns mean precision/recall + mAP
    P = float(val_res.box.mp) * 100.0
    R = float(val_res.box.mr) * 100.0
    mAP50 = float(val_res.box.map50) * 100.0
    mAP5095 = float(val_res.box.map) * 100.0

    # ---- Load metrics ----
    # Preload GT map once for speed
    gt_map = None

    # Overload@50 uses tau_load
    load_metrics, gt_map = det_fp_overload(images_dir, labels_dir, TAU_LOAD, gt_map=gt_map)
    overload = 0.0
    # Recompute overload using the per-image counts from tau_load (we stored only aggregates),
    # so do a fast second pass but ONLY counting dets (no IoU work).
    img_paths = list_images(images_dir)
    det_counts = []
    for p in tqdm(img_paths, desc="overload pass", leave=False):
        img = Image.open(p).convert("RGB")
        pred = model.predict(source=np.array(img), conf=TAU_LOAD, iou=NMS_IOU, imgsz=IMGSZ, verbose=False)[0]
        n = 0 if (pred.boxes is None) else len(pred.boxes)
        det_counts.append(n)
    det_counts = np.array(det_counts)
    overload = 100.0 * float((det_counts >= OVERLOAD_K).mean())

    out_rows = []

    # rows for each tau you care about
    for tau in TAUS:
        m, gt_map = det_fp_overload(images_dir, labels_dir, tau, gt_map=gt_map)
        out_rows.append({
            "dataset": name,
            "tau": tau,
            "images": m["images"],
            "P (%)": P,
            "R (%)": R,
            "mAP50 (%)": mAP50,
            "mAP50-95 (%)": mAP5095,
            "Det/img mean": m["Det/img mean"],
            "Det/img std":  m["Det/img std"],
            "FP/img mean":  m["FP/img mean"],
            "FP/img std":   m["FP/img std"],
            "Total det":    m["Total det"],
            "Total FP":     m["Total FP"],
            "Overload@50@0.001 (%)": overload,
            "Det/img@0.001 mean": float(det_counts.mean()),
            "Det/img@0.001 p95": float(np.percentile(det_counts, 95)),
        })

    return out_rows

# ---------------------------
# RUN ALL
# ---------------------------
rows = []
for name, root in DATASETS:
    print("\n==============================")
    print("DATASET:", name)
    print("ROOT:", root)
    rows += eval_dataset(name, root)

df = pd.DataFrame(rows).sort_values(["tau", "dataset"]).reset_index(drop=True)
display(df)

out_csv = "/content/paper_consistent_defended_eval.csv"
df.to_csv(out_csv, index=False)
print("\n✅ Saved:", out_csv)